# 14 · Historical catalog V3 (research only)

Exact raw description recovery for the accepted V2 retained events, a province × month retrospective history and the notebook-19 **selected**
spatial history with bounded, strict, as-of query functions for a later API.

* Inputs: manager pins `workflow/catalog/input-pins-v3.json`; every hash must also be recorded by its accepted producer (notebook 10 inputs /
  data_outputs, notebook 19 `data_files_gitignored`, 19 independent review). Raw CSVs are checked against the raw source manifest and the
  notebook-10 `v1_raw_csv` record (sha, bytes, encoding, years, record count, header).
* Recovery: `event_uid → row_lineage (source_file, source_row) → raw record`, required to satisfy `BE_year:norm(ACC_CODE) == event_uid`.
  Only `accident_type`, `presumed_cause`, `weather`; dates/time blocks/provinces stay the accepted repaired V2 values. Blank stays unknown.
* Descriptions are historical context only — never a prospective model feature, a causal explanation or personalised advice.
* Spatial as-of rule: issue = target_date − 1 day 18:00 Asia/Bangkok; a record counts only if its target end (day end for daily, block end
  for 6 h) ≤ issue − 30 days. Empty history (N = 0) is not evidence of safety.
* `production_ready = false`; no model is fitted or scored here; no HTTP API.


In [ ]:
# PARAMETERS: roots come from the runner environment (ROADAWARE_BACKEND_ROOT / ROADAWARE_V3_OUTPUT_ROOT); interactive default = this repo's backend/
# with the canonical outputs reports/v3/historical_catalog and data/v3/historical_catalog (refused if they already exist).
import os
from pathlib import Path


def _default_backend(c):
    for p in (c, *c.parents):
        if (p / 'config' / 'catalog_v3.json').is_file():
            return p
        if (p / 'backend' / 'config' / 'catalog_v3.json').is_file():
            return p / 'backend'
    return c


BACKEND = Path(os.environ.get('ROADAWARE_BACKEND_ROOT') or _default_backend(Path.cwd())).resolve()
REPO = BACKEND.parent
OUTPUT_ROOT = Path(os.environ.get('ROADAWARE_V3_OUTPUT_ROOT') or BACKEND).resolve()
CONFIG_PATH = BACKEND / 'config' / 'catalog_v3.json'
print('backend', BACKEND, '| output root', OUTPUT_ROOT)


In [ ]:
# FUNCTIONS (1/5): errors, hashing, strict JSON/CSV, output guards, protected snapshots, manager-pin and producer-membership verification and
# notebook cell hashing. Pure logic; the tests and the later API exec these cells (runtime never imports handoff files or tests).
import datetime as dt
import hashlib
import importlib
import importlib.util
import json
import math
import os
import platform
import re
import time
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd

NL = chr(10)
CATALOG_PRODUCER = '14_backend_catalog'
CONFIG_SCHEMA = 'roadaware-historical-catalog-v3-config/1'
MANIFEST_SCHEMA = 'roadaware-historical-catalog-v3-manifest/1'
DESC_FIELDS = ('accident_type', 'presumed_cause', 'weather')
CATEGORIES = DESC_FIELDS + ('time_block',)
TIME_BLOCKS = ('00-06', '06-12', '12-18', '18-24')
UNITS = ('daily', '6h')
N_FUNCTIONS_CELLS = 5
# files load_catalog requires (all hash-verified against the manifest before use)
REPORT_FILES_REQUIRED = ('province_monthly.csv', 'province_category_monthly.csv')
DATA_FILES_REQUIRED = ('provinces.json', 'universe_segment_ids.csv', 'selected_history_rows.csv.gz')
# JSON inputs whose content is used (strictly parsed); the 09 sources manifest is hash- and membership-checked only
JSON_DOCS = ('preparation_manifest', 'spatial_preparation_manifest', 'spatial_review', 'raw_manifest', 'data_pipeline')


class CatalogError(RuntimeError):
    pass


def utcnow():
    return dt.datetime.now(dt.timezone.utc).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


def atomic_write_bytes(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name('.' + path.name + '.tmp-' + str(os.getpid()))
    with open(tmp, 'xb') as f:  # exclusive: never follows or reuses an existing temp path
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def jsonable(o):
    # numpy scalars to Python, non-finite floats to None, so json.dumps(allow_nan=False) stays strict
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set, frozenset)):
        return [jsonable(v) for v in (sorted(o) if isinstance(o, (set, frozenset)) else o)]
    if isinstance(o, np.ndarray):
        return [jsonable(v) for v in o.tolist()]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    if o is pd.NA or o is pd.NaT:
        return None
    return o


def strict_json_bytes(obj):
    return (json.dumps(jsonable(obj), indent=2, ensure_ascii=False, allow_nan=False) + NL).encode('utf-8')


def canonical_sha(obj):
    return sha256_bytes(json.dumps(jsonable(obj), sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False).encode('utf-8'))


def atomic_write_json(path, obj):
    atomic_write_bytes(path, strict_json_bytes(obj))


def _no_duplicate_keys(pairs):
    out = {}
    for k, v in pairs:
        if k in out:
            raise CatalogError('duplicate JSON key ' + repr(k)[:80])
        out[k] = v
    return out


def _no_constant(c):
    raise CatalogError('non-finite JSON constant ' + c)


def strict_json_loads(text):
    """json.loads that refuses duplicate keys and NaN/Infinity constants."""
    return json.loads(text, object_pairs_hook=_no_duplicate_keys, parse_constant=_no_constant)


def read_strict_json(path):
    try:
        return strict_json_loads(Path(path).read_text('utf-8'))
    except CatalogError as e:
        raise CatalogError(str(e) + ' in ' + str(path))


def csv_bytes(df):
    num = df.select_dtypes(include=[np.number])
    if len(num.columns) and not np.isfinite(num.to_numpy(dtype=float)).all():
        raise CatalogError('refusing to write non-finite values to CSV')
    return df.to_csv(index=False, lineterminator=NL).encode('utf-8')


def read_str_csv(path, **kw):
    return pd.read_csv(path, dtype=str, keep_default_na=False, **kw)


def module_version(name):
    if importlib.util.find_spec(name) is None:
        return None
    return str(getattr(importlib.import_module(name), '__version__', 'unknown'))


def require_modules(names, pins):
    missing = [n for n in names if importlib.util.find_spec(n) is None]
    if missing:
        raise CatalogError('missing Python package(s) ' + ', '.join(missing) + '; ask Codex to install ' + ' '.join(pins.get(n, n) for n in missing))
    return True


def environment():
    return {'python': platform.python_version(), 'platform': platform.platform(),
            'dependencies': {m: module_version(m) for m in ('pandas', 'numpy', 'matplotlib')}}


def validate_config(cfg):
    if cfg.get('schema') != CONFIG_SCHEMA:
        raise CatalogError('catalog config schema is not ' + CONFIG_SCHEMA)
    if cfg.get('production_ready') is not False or cfg.get('research_only') is not True:
        raise CatalogError('catalog config must stay research-only (production_ready false)')
    if tuple(cfg['recovery']['fields']) != DESC_FIELDS or tuple(cfg['catalog']['time_blocks']) != TIME_BLOCKS:
        raise CatalogError('catalog config fields/time blocks differ from the notebook constants')
    if tuple(cfg['catalog']['categories']) != CATEGORIES:
        raise CatalogError('catalog config categories differ from the notebook constants')
    q = cfg['query']
    if int(q['latency_days']) != 30 or int(q['issue_hour_local']) != 18 or int(q['issue_day_offset']) != -1:
        raise CatalogError('query cutoff rule differs from the notebook-20 research latency assumption (issue D-1 18:00, 30 days)')
    if int(cfg['catalog']['utc_offset_seconds']) != 25200 or cfg['catalog']['timezone'] != 'Asia/Bangkok':
        raise CatalogError('catalog timezone must be Asia/Bangkok (UTC+7)')
    for k in ('runtime_output_paths', 'runtime_output_prefixes'):
        if not isinstance(cfg.get(k), list) or not cfg[k] or not all(isinstance(v, str) and v for v in cfg[k]):
            raise CatalogError('catalog config ' + k + ' must be a non-empty list of strings')
    return True


# ---------- output guards ----------
def output_paths(out_root, cfg):
    out_root = Path(os.path.abspath(out_root))
    rep, data = out_root / cfg['outputs']['report_dir'], out_root / cfg['outputs']['data_dir']
    return {'out_root': out_root, 'report_dir': rep, 'figure_dir': rep / 'figures', 'data_dir': data}


def _same(a, b):
    try:
        return os.path.samefile(a, b)
    except OSError:
        return False


def _inside(child, parent):
    c, p = Path(child).resolve(), Path(parent).resolve()
    if c == p or p in c.parents:
        return True
    return p.exists() and any(_same(a, p) for a in (c, *c.parents) if a.exists())


def symlinked_components(target, stop):
    """Symlinked path components of target strictly below stop (absolute, unresolved)."""
    t, s = Path(os.path.abspath(target)), Path(os.path.abspath(stop))
    out = []
    for a in (t, *t.parents):
        if a == s or a == a.parent:
            break
        if os.path.islink(a):
            out.append(str(a))
    return out


def input_rels(cfg):
    return [v['path'] for v in cfg['inputs'].values()]


def _rel_overlap(a, b):
    # backend-relative paths overlap when one is an ancestor of (or equal to) the other, compared by whole path components
    pa, pb = PurePosixPath(a).parts, PurePosixPath(b).parts
    n = min(len(pa), len(pb))
    return pa[:n] == pb[:n]


def runtime_output_rels(cfg):
    """Backend-relative paths that legitimately change while notebook 14 runs: the concurrent run roots and their report/data dirs, this
    catalog's own report/data dirs and the executing notebook-14 source (hashed separately). Never part of the protected snapshot."""
    return sorted(set(cfg['runtime_output_paths']) | {cfg['outputs']['report_dir'], cfg['outputs']['data_dir'], cfg['notebook']})


def snapshot_conflicts(cfg):
    """Snapshot paths that overlap a runtime output (either direction) or sit in a rerun-* root; [] for a sound config."""
    out = []
    for s in sorted(set(cfg['protected_paths']) | set(cfg['readonly_hashed_paths']) | set(input_rels(cfg))):
        if any(PurePosixPath(s).parts[0].startswith(p) for p in cfg['runtime_output_prefixes']):
            out.append(s + ' is inside a run root (' + str(cfg['runtime_output_prefixes']) + ')')
        out += [s + ' overlaps runtime output ' + r for r in runtime_output_rels(cfg) if _rel_overlap(s, r)]
    return out


def snapshot_rels(cfg):
    """Protected accepted inputs/outputs + read-only notebook-20 SOURCE files, hashed before and after the run. Fails closed if the config
    would snapshot a runtime output (those change concurrently and would make the before/after comparison meaningless)."""
    bad = snapshot_conflicts(cfg)
    if bad:
        raise CatalogError('protected snapshot overlaps runtime outputs: ' + '; '.join(bad[:10]))
    return sorted(set(cfg['protected_paths']) | set(cfg['readonly_hashed_paths']) | set(input_rels(cfg)))


def guarded_paths(backend, repo, cfg):
    backend, repo = Path(backend), Path(repo)
    rels = set(snapshot_rels(cfg)) | set(cfg['write_guard_paths']) | {cfg['notebook'], 'config/catalog_v3.json'}
    return [backend / r for r in sorted(rels)] + [repo / r for r in cfg['repo_write_guard_paths']] + [repo / cfg['pins']['path']]


def check_outputs(paths, backend, repo, cfg):
    """Call before any mkdir/write: report/data dirs must not overlap (either direction, symlinks/aliases resolved) any protected, read-only,
    pinned, concurrently-running or source path, must not overlap each other, must have no symlinked component below the backend/output
    root and must be NEW (an existing catalog is never overwritten)."""
    hits = []
    for label in ('report_dir', 'data_dir'):
        for p in guarded_paths(backend, repo, cfg):
            if _inside(paths[label], p) or _inside(p, paths[label]):
                hits.append(label + ' ' + str(paths[label]) + ' overlaps guarded ' + str(p))
        stop = backend if _inside(paths[label], backend) else paths['out_root']
        for s in symlinked_components(paths[label], stop) + ([str(paths['out_root'])] if os.path.islink(paths['out_root']) else []):
            hits.append(label + ' has a symlinked component ' + s)
    if _inside(paths['report_dir'], paths['data_dir']) or _inside(paths['data_dir'], paths['report_dir']):
        hits.append('report and data dirs overlap')
    if hits:
        raise CatalogError('refusing output root (nothing written): ' + '; '.join(hits))
    used = [str(paths[k]) for k in ('report_dir', 'data_dir') if os.path.lexists(paths[k])]
    if used:
        raise CatalogError('output already exists in ' + ', '.join(used) + '; use a NEW root, e.g. backend/rerun-historical-catalog-<id>')
    return True


def check_report_file(rel, cfg):
    if Path(rel).suffix.lower() not in cfg['outputs']['report_allowed_suffixes'] or str(rel).endswith('.gz'):
        raise CatalogError('refusing to write ' + str(rel) + ' into the report dir (aggregated csv/png/md/json only)')
    return True


def hash_paths(root, rels):
    root, out = Path(root), {}
    for rel in rels:
        p = root / rel
        if os.path.islink(p):
            out[Path(rel).as_posix()] = 'symlink:' + os.readlink(p)
        elif p.is_file():
            out[Path(rel).as_posix()] = sha256_file(p)
        elif p.is_dir():
            for q in sorted(p.rglob('*')):
                if q.is_symlink():
                    out[(Path(rel) / q.relative_to(p)).as_posix()] = 'symlink:' + os.readlink(q)
                elif q.is_file():
                    out[(Path(rel) / q.relative_to(p)).as_posix()] = sha256_file(q)
        else:
            out[Path(rel).as_posix()] = None
    return out


def snapshot_diff(before, after):
    return sorted(k for k in set(before) | set(after) if before.get(k) != after.get(k))


# ---------- manager pins and producer membership ----------
def verify_pinned(path, expected, label):
    path = Path(path)
    if not expected:
        raise CatalogError(label + ': no sha256 pin (fail closed)')
    if os.path.islink(path) or not path.is_file():
        raise CatalogError(label + ': missing or non-regular input ' + str(path))
    got = sha256_file(path)
    if got != expected:
        raise CatalogError(label + ': sha256 ' + got[:12] + ' != pinned ' + str(expected)[:12] + ' (' + str(path) + '); input changed or tampered')
    return got


def load_pins(repo, cfg):
    """Manager pins (repository-relative path from the config). The pinned file set must equal the config input set exactly."""
    p = Path(repo) / cfg['pins']['path']
    pins = read_strict_json(p)
    if pins.get('schema') != cfg['pins']['schema'] or pins.get(cfg['pins']['registered_flag']) is not True:
        raise CatalogError('manager pins have the wrong schema or are not registered before the catalog implementation')
    files = pins.get('files')
    if not isinstance(files, dict) or set(files) != set(input_rels(cfg)):
        raise CatalogError('manager pin file set differs from the config inputs: ' + str(sorted(set(files or {}) ^ set(input_rels(cfg)))))
    return pins, sha256_file(p)


def _path_suffix_match(recorded, rel):
    # producer manifests record absolute original paths (provenance); membership is the backend-relative suffix, never the absolute root
    r = str(recorded).replace('\\', '/')
    return r == rel or r.endswith('/' + rel)


def producer10_input_sha(m10, role, rel):
    hits = [x for x in m10.get('inputs', []) if x.get('role') == role and _path_suffix_match(x.get('path', ''), rel)]
    if len(hits) != 1:
        raise CatalogError('notebook-10 manifest has ' + str(len(hits)) + ' inputs with role ' + role + ' for ' + rel + ' (need exactly one)')
    return hits[0]['sha256']


def corroborate(name, spec, sha, path, docs, cfg):
    """One accepted-producer membership rule; returns a short evidence string or raises."""
    rule, P = spec['rule'], cfg['producers']
    m10, m19 = docs.get('preparation_manifest'), docs.get('spatial_preparation_manifest')
    if rule == 'producer10_self':
        if m10.get('producer') != P['10']['producer'] or m10.get('schema') != P['10']['schema'] or m10.get('status') != P['10']['status']:
            raise CatalogError('notebook-10 manifest is not the accepted completed producer')
        return 'producer 10 ' + m10['status']
    if rule == 'producer10_input':
        if producer10_input_sha(m10, spec['role'], cfg['inputs'][name]['path']) != sha:
            raise CatalogError(name + ': sha differs from the notebook-10 recorded ' + spec['role'] + ' input')
        return '10 input role ' + spec['role']
    if rule == 'producer10_data_output':
        if Path(cfg['inputs'][name]['path']).parent.as_posix() != P['10']['data_dir']:
            raise CatalogError(name + ': not inside the notebook-10 data dir')
        hits = [x for x in m10.get('data_outputs', []) if x.get('path') == spec['name']]
        if len(hits) != 1 or hits[0].get('sha256') != sha or int(hits[0].get('bytes', -1)) != Path(path).stat().st_size:
            raise CatalogError(name + ': not an exact notebook-10 data_outputs member (sha/bytes)')
        return '10 data_outputs ' + spec['name']
    if rule == 'producer19_self':
        if m19.get('status') != P['19']['status']:
            raise CatalogError('notebook-19 manifest status is ' + str(m19.get('status')) + ', not complete')
        return 'producer 19 complete'
    if rule == 'producer19_data':
        if Path(cfg['inputs'][name]['path']).parent.as_posix() != P['19']['data_dir']:
            raise CatalogError(name + ': not inside the notebook-19 data dir')
        if (m19.get('data_files_gitignored') or {}).get(spec['name']) != sha:
            raise CatalogError(name + ': sha differs from notebook-19 data_files_gitignored')
        return '19 data_files_gitignored ' + spec['name']
    if rule == 'review19':
        rv = docs['spatial_review']
        if rv.get('status') != P['19_review']['status']:
            raise CatalogError('notebook-19 independent review status is ' + str(rv.get('status')))
        return 'review ' + rv['status']
    raise CatalogError('unknown corroboration rule ' + str(rule))


def verify_inputs(backend, cfg, pins):
    """Hash-verify every pinned input, then corroborate each hash with its accepted producer (10 inputs/data_outputs, 19 data_files_gitignored,
    19 review). Returns (rows DataFrame, parsed JSON docs)."""
    backend, rows, docs, shas = Path(backend), [], {}, {}
    for name, v in cfg['inputs'].items():
        p = backend / v['path']
        shas[name] = verify_pinned(p, pins['files'][v['path']], name)
        if name in JSON_DOCS:
            docs[name] = read_strict_json(p)
    for name, v in cfg['inputs'].items():
        ev = corroborate(name, v['corroborate'], shas[name], backend / v['path'], docs, cfg)
        rows.append({'input': name, 'path': v['path'], 'sha256': shas[name], 'pin': 'manager pin', 'producer_membership': ev, 'status': 'pass'})
    return pd.DataFrame(rows), docs


def _safe_rel(rel, label):
    pp = PurePosixPath(str(rel))
    if not rel or pp.is_absolute() or any(part in ('..', '.', '') for part in pp.parts) or '\\' in str(rel):
        raise CatalogError(label + ': raw manifest path must be backend-relative without ..: ' + repr(rel))
    return pp


def verify_raw_sources(backend, cfg, raw_manifest, m10, needed_files):
    """Each needed raw CSV: exactly one raw-manifest entry by file name, backend-relative path (the recorded original_path is provenance only),
    sha equal to the raw manifest AND to the notebook-10 v1_raw_csv input, byte size, allowed encoding, integer years/rows. Returns {file: info}."""
    backend, out = Path(backend), {}
    files = raw_manifest.get('files')
    if not isinstance(files, list):
        raise CatalogError('raw source manifest has no files list')
    for f in sorted(needed_files):
        hits = [e for e in files if e.get('file') == f]
        if len(hits) != 1:
            raise CatalogError('raw source ' + str(f) + ': ' + str(len(hits)) + ' raw-manifest entries (need exactly one)')
        e = hits[0]
        pp = _safe_rel(e.get('path'), f)
        if pp.name != f:
            raise CatalogError('raw source ' + f + ': manifest path name differs from file')
        p = backend.joinpath(*pp.parts)
        if symlinked_components(p, backend):
            raise CatalogError('raw source ' + f + ': symlinked path component')
        got = verify_pinned(p, e.get('sha256'), 'raw ' + f)
        if producer10_input_sha(m10, cfg['raw']['producer10_role'], pp.as_posix()) != got:
            raise CatalogError('raw ' + f + ': sha differs from the notebook-10 v1_raw_csv input')
        if 'bytes' in e and int(e['bytes']) != p.stat().st_size:
            raise CatalogError('raw ' + f + ': byte size differs from the raw manifest')
        if e.get('encoding') not in cfg['raw']['allowed_encodings']:
            raise CatalogError('raw ' + f + ': encoding ' + repr(e.get('encoding')) + ' not allowed')
        for k in ('be_year', 'ce_year', 'rows'):
            if type(e.get(k)) is not int or e[k] < 0:
                raise CatalogError('raw ' + f + ': manifest ' + k + ' is not a non-negative integer')
        if e['be_year'] - e['ce_year'] != 543:
            raise CatalogError('raw ' + f + ': be_year/ce_year inconsistent')
        out[f] = {'path': p, 'rel': pp.as_posix(), 'entry': e, 'sha256': got}
    return out


# ---------- notebook cell hashing ----------
def notebook_code_cells(nb_path):
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    out = []
    for i, c in enumerate(nb['cells']):
        if c['cell_type'] == 'code':
            out.append((i, str(c.get('id', '')), ''.join(c['source']) if isinstance(c['source'], list) else c['source']))
    return out


def functions_sha(cells, expected_cells=N_FUNCTIONS_CELLS):
    """sha256 of the '\\n\\n'-joined '# FUNCTIONS' code cells (the notebook-19/20 convention)."""
    funcs = [src for _, _, src in cells if src.lstrip().startswith('# FUNCTIONS')]
    if len(funcs) != int(expected_cells):
        raise CatalogError('expected ' + str(expected_cells) + ' FUNCTIONS cells, found ' + str(len(funcs)))
    return sha256_bytes('\n\n'.join(funcs).encode('utf-8'))


def code_cells_sha(cells):
    return {str(i) + ':' + cid: sha256_bytes(src.encode('utf-8')) for i, cid, src in cells}


In [ ]:
# FUNCTIONS (2/5): exact raw description recovery. Accepted V2 retained events -> row_lineage source_file/source_row -> the hash-verified raw
# CSV record, checked by exact BE-year:ACC_CODE equality. Only accident_type / presumed_cause / weather are read; dates, time blocks and
# provinces always come from the accepted (repaired) V2 events and are never re-parsed from raw. No fuzzy, id-only or guessed joins.
EVENT_COLS = ('event_uid', 'source_year_ce', 'event_date', 'time_sec', 'time_block', 'province_key')
LINEAGE_COLS = ('event_uid', 'source_file', 'source_year_ce', 'source_row', 'v2_disposition', 'event_date_v2', 'time_sec', 'province_key')
INT_RE = re.compile(r'[0-9]+')
DATE_RE = re.compile(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')


def norm_text(s):
    # identical to the V1 notebook-03 norm_text (whitespace runs -> one space, strip; blanks stay '')
    return s.fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()


def require_columns(df, cols, label):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise CatalogError(label + ': missing columns ' + str(miss))
    return True


def resolve_alias(columns, aliases, field, required=False):
    """Exactly one present audited alias per field: two present -> ambiguous (fail); none -> None (explicit unknown) unless required."""
    names = aliases.get(field)
    if not isinstance(names, list) or not names or not all(isinstance(n, str) for n in names):
        raise CatalogError('schema_aliases has no alias list for ' + field)
    present = [c for c in names if c in list(columns)]
    if len(present) > 1:
        raise CatalogError('ambiguous headers for ' + field + ': ' + str(present))
    if not present:
        if required:
            raise CatalogError('required field ' + field + ' has no header among ' + str(names))
        return None
    return present[0]


def read_raw_csv(path, entry):
    """Read one verified raw CSV as strings with the manifest encoding; record count and header must equal the raw manifest."""
    cols = entry.get('columns')
    if cols is not None and (not isinstance(cols, list) or len(set(cols)) != len(cols)):
        raise CatalogError('raw ' + entry['file'] + ': manifest header list has duplicates (ambiguous headers)')
    raw = read_str_csv(path, encoding=entry['encoding'])
    if len(raw) != int(entry['rows']):
        raise CatalogError('raw ' + entry['file'] + ': ' + str(len(raw)) + ' records != manifest rows ' + str(entry['rows']))
    if cols is not None and list(raw.columns) != cols:
        raise CatalogError('raw ' + entry['file'] + ': header differs from the raw manifest columns')
    if 'n_columns' in entry and len(raw.columns) != int(entry['n_columns']):
        raise CatalogError('raw ' + entry['file'] + ': column count differs from the raw manifest')
    return raw


def join_retained_lineage(events, lineage, cfg):
    """1:1 retained lineage <-> accepted events. Requires unique UIDs, equal UID sets, unique integer source coordinates and agreement of the
    lineage's recorded V2 date/time/province/year with the accepted events (repaired values are taken from events, never recomputed)."""
    require_columns(events, EVENT_COLS, 'events_v2')
    require_columns(lineage, LINEAGE_COLS, 'row_lineage_v2')
    E = events[list(EVENT_COLS)].copy()
    L = lineage[lineage['v2_disposition'] == cfg['recovery']['retained_disposition']][list(LINEAGE_COLS)].copy()
    if E['event_uid'].duplicated().any():
        raise CatalogError('events_v2 event_uid not unique')
    if L['event_uid'].duplicated().any():
        raise CatalogError('retained lineage event_uid not unique')
    if set(E['event_uid']) != set(L['event_uid']):
        raise CatalogError('retained lineage UID set != events UID set (only_events=' + str(len(set(E['event_uid']) - set(L['event_uid']))) +
                           ', only_lineage=' + str(len(set(L['event_uid']) - set(E['event_uid']))) + ')')
    if not L['source_row'].str.fullmatch(INT_RE).all():
        raise CatalogError('retained lineage source_row is not a plain non-negative integer everywhere')
    if L.duplicated(['source_file', 'source_row']).any():
        raise CatalogError('duplicate retained source coordinates (source_file, source_row)')
    J = E.merge(L.rename(columns={'time_sec': 'time_sec_lineage', 'province_key': 'province_key_lineage', 'source_year_ce': 'source_year_ce_lineage'}),
                on='event_uid', how='inner', validate='one_to_one')
    if len(J) != len(E):
        raise CatalogError('lineage join lost events')
    bad = {'source_year_ce': int((J['source_year_ce'] != J['source_year_ce_lineage']).sum()),
           'event_date': int((J['event_date'] != J['event_date_v2']).sum()),
           'province_key': int((J['province_key'] != J['province_key_lineage']).sum())}
    a, b = pd.to_numeric(J['time_sec'], errors='coerce'), pd.to_numeric(J['time_sec_lineage'], errors='coerce')
    bad['time_sec'] = int((~((a == b) | (a.isna() & b.isna()))).sum())
    if any(bad.values()):
        raise CatalogError('accepted events disagree with their lineage record: ' + str(bad))
    if not J['event_date'].str.fullmatch(DATE_RE).all():
        raise CatalogError('accepted event_date is not ISO YYYY-MM-DD everywhere')
    J['source_row_int'] = J['source_row'].astype('int64')
    return J.drop(columns=['event_date_v2', 'time_sec_lineage', 'province_key_lineage', 'source_year_ce_lineage', 'v2_disposition'])


def recover_descriptions(joined, raw_sources, aliases, raw_loader, fields=DESC_FIELDS, first_row=2):
    """For every joined event read its exact raw record (iloc = source_row - first_row) and require str(be_year)+':'+norm(ACC_CODE) ==
    event_uid. raw_loader(file) returns the verified raw DataFrame. Returns (recovered frame, per-file provenance rows). No raw-only rows."""
    parts, prov = [], []
    unknown_files = sorted(set(joined['source_file']) - set(raw_sources))
    if unknown_files:
        raise CatalogError('lineage references raw files outside the verified raw sources: ' + str(unknown_files))
    for f, g in joined.groupby('source_file', sort=True):
        e = raw_sources[f]['entry']
        if (g['source_year_ce'] != str(e['ce_year'])).any():
            raise CatalogError('raw ' + f + ': event source_year_ce differs from the raw manifest ce_year ' + str(e['ce_year']))
        raw = raw_loader(f)
        n = len(raw)
        rows = g['source_row_int'].to_numpy()
        if (rows < first_row).any() or (rows > n + first_row - 1).any():
            raise CatalogError('raw ' + f + ': source_row outside [' + str(first_row) + ', ' + str(n + first_row - 1) + ']')
        r = raw.iloc[rows - first_row].reset_index(drop=True)
        idcol = resolve_alias(raw.columns, aliases, 'acc_code', required=True)
        expected = (str(e['be_year']) + ':' + norm_text(r[idcol])).to_numpy()
        mism = int((expected != g['event_uid'].to_numpy()).sum())
        if mism:
            raise CatalogError('raw ' + f + ': ' + str(mism) + ' rows where BE-year:ACC_CODE != event_uid (row/UID mismatch)')
        out = g.reset_index(drop=True).copy()
        headers = {}
        for fld in fields:
            col = resolve_alias(raw.columns, aliases, fld)
            headers[fld] = col
            out[fld] = norm_text(r[col]) if col is not None else ''
        out['ce_year'] = int(e['ce_year'])
        parts.append(out)
        prov.append({'source_file': f, 'be_year': e['be_year'], 'ce_year': e['ce_year'], 'raw_sha256': raw_sources[f]['sha256'], 'raw_records': n,
                     'events_recovered': len(out), 'id_header': idcol, **{'header_' + k: (v if v is not None else '') for k, v in headers.items()},
                     **{'missing_header_' + k: v is None for k, v in headers.items()}})
    rec = pd.concat(parts, ignore_index=True)
    if len(rec) != len(joined) or rec['event_uid'].duplicated().any() or set(rec['event_uid']) != set(joined['event_uid']):
        raise CatalogError('recovered rows are not exactly the accepted events (no new raw-only records allowed)')
    return rec.sort_values('event_uid', kind='stable').reset_index(drop=True), pd.DataFrame(prov)


def compare_legacy(rec, legacy, fields=DESC_FIELDS):
    """Every legacy V1 descriptor UID that is an accepted event must agree exactly on all three fields (fail on any disagreement). Adds
    legacy_status (legacy_matched / repaired_recovered). Returns (rec, per-field coverage rows, summary)."""
    require_columns(legacy, ('event_uid',) + tuple(fields), 'legacy descriptive file')
    if legacy['event_uid'].duplicated().any():
        raise CatalogError('legacy descriptive event_uid not unique')
    j = rec.merge(legacy[['event_uid'] + list(fields)].rename(columns={f: f + '_legacy' for f in fields}), on='event_uid', how='left',
                  validate='one_to_one', indicator=True)
    known = j['_merge'].eq('both').to_numpy()
    conflicts = {f: int((j.loc[known, f] != j.loc[known, f + '_legacy']).sum()) for f in fields}
    if any(conflicts.values()):
        raise CatalogError('legacy descriptor conflicts with exact raw recovery: ' + str(conflicts))
    out = rec.copy()
    out['legacy_status'] = np.where(known, 'legacy_matched', 'repaired_recovered')
    cov = []
    for f in fields:
        before = (~known) | (j[f + '_legacy'].fillna('') == '').to_numpy()
        after = (j[f] == '').to_numpy()
        cov.append({'field': f, 'events': len(j), 'legacy_matched_events': int(known.sum()), 'legacy_value_conflicts': conflicts[f],
                    'unknown_before_recovery': int(before.sum()), 'unknown_after_recovery': int(after.sum()),
                    'known_after_recovery': int((~after).sum()), 'newly_known_from_raw': int((before & ~after).sum()),
                    'blank_in_raw_among_repaired': int((~known & after).sum()), 'blank_in_raw_among_legacy': int((known & after).sum())})
    summary = {'events': len(j), 'legacy_matched': int(known.sum()), 'repaired_recovered': int((~known).sum()),
               'legacy_uids_not_accepted_events': int(len(set(legacy['event_uid']) - set(rec['event_uid'])))}
    return out, pd.DataFrame(cov), summary


def recovery_by_year(rec, fields=DESC_FIELDS):
    rows = []
    for y, g in rec.groupby('ce_year', sort=True):
        r = {'ce_year': int(y), 'events': len(g), 'legacy_matched': int((g['legacy_status'] == 'legacy_matched').sum()),
             'repaired_recovered': int((g['legacy_status'] == 'repaired_recovered').sum())}
        for f in fields:
            r['unknown_' + f] = int((g[f] == '').sum())
        rows.append(r)
    return pd.DataFrame(rows)


In [ ]:
# FUNCTIONS (3/5): province x month history (all 106308 accepted retained records) and notebook-19 SELECTED spatial history (selected +
# 50 m linked + assigned records only; conditional, never a complete all-road count). A blank description / time block is the explicit
# category 'unknown'; it is never allocated or imputed. Every table reconciles to its denominators.
SELECTED_COLS = ('event_uid', 'ce_year', 'event_date', 'time_block', 'province_key', 'segment_id_2000m')
SPATIAL_ROW_COLS = ('segment_id', 'event_date', 'time_block', 'ce_year') + DESC_FIELDS


def month_range(first, last):
    y, m = int(first[:4]), int(first[5:7])
    out = []
    while '%04d-%02d' % (y, m) <= last:
        out.append('%04d-%02d' % (y, m))
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out


def load_provinces(points, expected):
    require_columns(points, ('province_key', 'province_code', 'name_th', 'name_en', 'lat', 'lon', 'point_meaning'), 'province_points')
    if len(points) != int(expected) or points['province_key'].duplicated().any() or points['province_code'].duplicated().any():
        raise CatalogError('province points are not ' + str(expected) + ' unique provinces')
    P = points[['province_key', 'province_code', 'name_th', 'name_en', 'lat', 'lon', 'point_meaning']].copy()
    P['lat'], P['lon'] = P['lat'].astype(float), P['lon'].astype(float)
    return P.sort_values('province_code', key=lambda s: s.astype(int), kind='stable').reset_index(drop=True)


def check_event_domain(rec, provinces, months, blocks=TIME_BLOCKS):
    bad_p = sorted(set(rec['province_key']) - set(provinces['province_key']))
    if bad_p:
        raise CatalogError('events with province_key outside the 77 provinces: ' + str(bad_p[:5]))
    ym = rec['event_date'].str[:7]
    if not ym.isin(set(months)).all():
        raise CatalogError('events outside the catalog months ' + months[0] + '..' + months[-1])
    bad_b = sorted(set(rec['time_block']) - set(blocks) - {''})
    if bad_b:
        raise CatalogError('time_block values outside ' + str(blocks) + ': ' + str(bad_b[:5]))
    return True


def province_tables(rec, provinces, months, cats=CATEGORIES):
    """Full 77 x 60 grid (explicit zeros) with known/unknown denominators per category, and the long category table (nonzero cells)."""
    E = rec.assign(year_month=rec['event_date'].str[:7])
    grid = pd.MultiIndex.from_product([provinces['province_key'].tolist(), months], names=['province_key', 'year_month'])
    M = E.groupby(['province_key', 'year_month']).size().reindex(grid, fill_value=0).rename('n_records').reset_index()
    for c in cats:
        k = E[E[c] != ''].groupby(['province_key', 'year_month']).size().reindex(grid, fill_value=0).to_numpy()
        M['known_' + c] = k
        M['unknown_' + c] = M['n_records'].to_numpy() - k
    long = []
    for c in cats:
        g = E.groupby(['province_key', 'year_month', c]).size().rename('n_records').reset_index().rename(columns={c: 'value'})
        g.insert(2, 'category', c)
        g['label_status'] = np.where(g['value'] == '', 'unknown', 'known')
        long.append(g)
    C = pd.concat(long, ignore_index=True)[['province_key', 'year_month', 'category', 'value', 'label_status', 'n_records']]
    C = C.sort_values(['province_key', 'year_month', 'category', 'value'], kind='stable').reset_index(drop=True)
    return M, C


def reconcile_province(M, C, rec, provinces, months, cats=CATEGORIES):
    """Rows (check, expected, observed, ok) for grid size, total, per-year totals, known+unknown denominators and long-table marginals."""
    rows = [('grid rows == provinces x months', len(provinces) * len(months), len(M)),
            ('grid total == accepted events', len(rec), int(M['n_records'].sum())),
            ('zero province-months (zero selected records only)', None, int((M['n_records'] == 0).sum()))]
    yr = rec['event_date'].str[:4].value_counts()
    for y, n in sorted(M.assign(y=M['year_month'].str[:4]).groupby('y')['n_records'].sum().items()):
        rows.append(('year ' + y + ' grid == events', int(yr.get(y, 0)), int(n)))
    key = M.set_index(['province_key', 'year_month'])['n_records']
    for c in cats:
        rows.append(('known+unknown == records: ' + c, int(M['n_records'].sum()), int((M['known_' + c] + M['unknown_' + c]).sum())))
        s = C[C['category'] == c].groupby(['province_key', 'year_month'])['n_records'].sum().reindex(key.index, fill_value=0)
        rows.append(('category cells == grid cells: ' + c, len(key), int((s == key).sum())))
        u = C[(C['category'] == c) & (C['label_status'] == 'unknown')]['n_records'].sum()
        rows.append(('long unknown == grid unknown: ' + c, int(M['unknown_' + c].sum()), int(u)))
    return check_rows(rows)


def check_rows(rows):
    # expected None = informational row (always ok); kept as '' so the CSV has no NaN
    return pd.DataFrame([{'check': c, 'expected': '' if e is None else int(e), 'observed': int(o), 'ok': bool(e is None or int(e) == int(o))}
                         for c, e, o in rows])


def spatial_rows(selected, rec, universe_ids, blocks=TIME_BLOCKS):
    """1:1 join of every notebook-19 selected UID to the recovered descriptions; accepted date/block/province/year must match exactly and
    segments must be universe members. Returns the event-free row context (no UID, source row or coordinates)."""
    require_columns(selected, SELECTED_COLS, 'selected_events')
    S = selected[list(SELECTED_COLS)].copy()
    if S['event_uid'].duplicated().any():
        raise CatalogError('selected event_uid not unique')
    extra = set(S['event_uid']) - set(rec['event_uid'])
    if extra:
        raise CatalogError(str(len(extra)) + ' selected UIDs are not accepted recovered events')
    J = S.merge(rec[['event_uid', 'event_date', 'time_block', 'province_key', 'ce_year'] + list(DESC_FIELDS)], on='event_uid', how='left',
                suffixes=('', '_acc'), validate='one_to_one')
    bad = {c: int((J[c] != J[c + '_acc']).sum()) for c in ('event_date', 'time_block', 'province_key')}
    bad['ce_year'] = int((J['ce_year'] != J['ce_year_acc'].astype(str)).sum())
    if any(bad.values()):
        raise CatalogError('selected events disagree with accepted events: ' + str(bad))
    if not J['time_block'].isin(set(blocks)).all():
        raise CatalogError('selected events with a time block outside ' + str(blocks))
    if not J['segment_id_2000m'].isin(set(universe_ids)).all():
        raise CatalogError('selected events on segments outside the universe')
    R = J.rename(columns={'segment_id_2000m': 'segment_id'})[list(SPATIAL_ROW_COLS)].copy()
    R['ce_year'] = R['ce_year'].astype(int)
    return R.sort_values(['segment_id', 'event_date', 'time_block'], kind='stable').reset_index(drop=True)


def sparse_counts(R):
    d = R.groupby(['segment_id', 'event_date']).size().rename('y_count').reset_index().rename(columns={'event_date': 'target_date'})
    b = R.groupby(['segment_id', 'event_date', 'time_block']).size().rename('y_count').reset_index().rename(columns={'event_date': 'target_date'})
    return d, b


def counts_identical(a, b, keys):
    """Exact equality of sparse count tables after canonical sort (y_count compared as integers)."""
    if set(a.columns) < set(keys) | {'y_count'} or set(b.columns) < set(keys) | {'y_count'}:
        return False
    x = a[list(keys) + ['y_count']].astype({'y_count': 'int64'}).astype({k: str for k in keys}).sort_values(list(keys)).reset_index(drop=True)
    y = b[list(keys) + ['y_count']].astype({'y_count': 'int64'}).astype({k: str for k in keys}).sort_values(list(keys)).reset_index(drop=True)
    return len(x) == len(y) and x.equals(y)


def spatial_tables(R, universe_ids, months, years, cats=CATEGORIES):
    """Monthly totals, category totals, per-segment summary (ALL universe ids, zero history explicit) and year coverage incl. source gaps."""
    X = R.assign(year_month=R['event_date'].str[:7])
    mon = X.groupby('year_month').size().reindex(months, fill_value=0).rename('n_selected').reset_index()
    mon['segments_with_records'] = X.groupby('year_month')['segment_id'].nunique().reindex(months, fill_value=0).to_numpy()
    for c in cats:
        mon['unknown_' + c] = X[X[c] == ''].groupby('year_month').size().reindex(months, fill_value=0).to_numpy()
    tot = []
    for c in cats:
        g = X.groupby(c).size().rename('n_selected').reset_index().rename(columns={c: 'value'})
        g.insert(0, 'category', c)
        g['label_status'] = np.where(g['value'] == '', 'unknown', 'known')
        g['share_of_selected'] = g['n_selected'] / max(len(X), 1)
        tot.append(g.sort_values(['n_selected', 'value'], ascending=[False, True], kind='stable'))
    seg = pd.DataFrame({'segment_id': sorted(universe_ids)})
    n = X.groupby('segment_id').size()
    seg['n_selected_total'] = seg['segment_id'].map(n).fillna(0).astype(int)
    seg['n_months_with_records'] = seg['segment_id'].map(X.groupby('segment_id')['year_month'].nunique()).fillna(0).astype(int)
    seg['history_status'] = np.where(seg['n_selected_total'] > 0, 'has_selected_history', 'empty_selected_history_not_safety')
    cov = []
    for y in years:
        g = X[X['ce_year'] == int(y)]
        in_src = '%04d-01' % int(y) >= months[0] and '%04d-12' % int(y) <= months[-1]
        r = {'year': int(y), 'catalog_source_coverage': 'covered' if in_src else 'no_source_data_gap', 'n_selected': len(g),
             'segments_with_records': int(g['segment_id'].nunique()), 'segments_without_records': len(universe_ids) - int(g['segment_id'].nunique())}
        for c in DESC_FIELDS:
            r['unknown_' + c] = int((g[c] == '').sum())
        cov.append(r)
    return mon, pd.concat(tot, ignore_index=True), seg, pd.DataFrame(cov)


def reconcile_spatial(R, selected, universe_ids, counts_daily, counts_6h, review, mon, seg):
    d, b = sparse_counts(R)
    rows = [('selected rows == notebook-19 selected file', len(selected), len(R)),
            ('selected rows == 19 independent review', int(review.get('selected_events', -1)), len(R)),
            ('universe ids == 19 independent review', int(review.get('universe_segments', -1)), len(universe_ids)),
            ('universe ids unique', len(universe_ids), len(set(universe_ids))),
            ('daily recount identical to 19 counts_daily (1=yes)', 1, int(counts_identical(d, counts_daily, ('segment_id', 'target_date')))),
            ('6h recount identical to 19 counts_6h (1=yes)', 1, int(counts_identical(b, counts_6h, ('segment_id', 'target_date', 'time_block')))),
            ('counts_daily total == selected', len(R), int(pd.to_numeric(counts_daily['y_count']).sum())),
            ('counts_6h total == selected', len(R), int(pd.to_numeric(counts_6h['y_count']).sum())),
            ('monthly total == selected', len(R), int(mon['n_selected'].sum())),
            ('segment summary rows == universe', len(universe_ids), len(seg)),
            ('segment summary total == selected', len(R), int(seg['n_selected_total'].sum())),
            ('segments with empty selected history', None, int((seg['n_selected_total'] == 0).sum()))]
    return check_rows(rows)


In [ ]:
# FUNCTIONS (4/5): REUSABLE CATALOG INTERFACE (later API). load_catalog verifies manifest / FUNCTIONS / config / optional source-notebook and
# every output hash BEFORE reading any table; queries are strict (exact types, no extra or missing fields, bounded sizes) and return
# aggregates only (never event UIDs, source rows, coordinates or row-level records). Spatial history is as-of: the forecast issue time is
# target_date - 1 day at 18:00 Asia/Bangkok and only records whose TARGET END <= issue - 30 days count (day end for daily, block end for 6h;
# the notebook-20 research latency assumption). Province history is retrospective (all records in the month range), never an as-of forecast.
PROVINCE_SCHEMA = 'roadaware-province-history/1'
SPATIAL_SCHEMA = 'roadaware-spatial-history/1'
MONTH_RE = re.compile(r'[0-9]{4}-(0[1-9]|1[0-2])')
PROVINCE_KEYS = ('province_key', 'start_month', 'end_month', 'categories', 'top_k')
PROVINCE_REQUIRED = ('province_key', 'start_month', 'end_month')
SPATIAL_KEYS = ('segment_id', 'target_date', 'unit', 'time_block', 'lookback_days', 'categories', 'top_k')
SPATIAL_REQUIRED = ('segment_id', 'target_date', 'unit')
MAX_STR = 200
RESEARCH_STATUS = {'research_only': True, 'production_ready': False, 'operational_status': 'BLOCKED', 'live_data': False, 'forecast': False}
PROVINCE_NOTE = ('RETROSPECTIVE historical count of accepted recorded MOT-source accident records (V2 repaired dates) in this province and month '
                 'range, using every record in the range; NOT an as-of forecast input. A zero month means zero recorded records in this source, '
                 'not complete all-road coverage. Province names/points are representative labels, never event locations.')
SPATIAL_NOTE = ('AS-OF historical count of notebook-19 SELECTED records (primary 50 m linked and assigned to this 2000 m segment); conditional on '
                'that selection, never a complete all-road count. Only records whose target end <= issue time - 30 days are used. An empty '
                'history (N = 0) means no selected records, NOT evidence of safety.')
CATEGORY_NOTE = ('Categories are historical recorded attributes only: presumed_cause is a recorded attribute, not proof of cause; weather is '
                 'the historical recorded weather, not current weather. Not causal explanations, not model inputs, not personalised advice.')


# ---------- explicit PUBLIC result schema (every key of a query answer; anything else is a schema violation) ----------
# Public stable segment ids, ISO dates/times, province labels, aggregate counts and recorded category labels are allowed. Event UIDs,
# source rows/files, ACC codes, event times, individual coordinates/geometry and narratives are forbidden as keys at ANY depth.
FORBIDDEN_OUTPUT_KEYS = frozenset({'event_uid', 'event_uids', 'uid', 'uids', 'source_row', 'source_rows', 'source_file', 'source_files', 'acc_code',
                                   'time_sec', 'latitude', 'longitude', 'lat', 'lon', 'x', 'y', 'coordinates', 'geometry', 'wkt', 'chosen_source_oid',
                                   'chosen_part', 'chosen_piece', 'narrative', 'narratives', 'description', 'raw', 'raw_row', 'rows', 'event_rows'})
RESEARCH_STATUS_KEYS = frozenset(RESEARCH_STATUS)
CATEGORY_BLOCK_KEYS = frozenset({'denominator_records', 'known_records', 'unknown_records', 'known_share', 'distinct_known_values', 'top_values',
                                 'other_known_records'})
TOP_VALUE_KEYS = frozenset({'value', 'records'})
MONTH_COUNT_KEYS = frozenset({'year_month', 'records'})
HISTORY_WINDOW_KEYS = frozenset({'lookback_days', 'window_start_local_exclusive', 'window_end_local_inclusive', 'catalog_data_start_local',
                                 'catalog_data_end_local_exclusive', 'covered_days', 'window_fully_inside_catalog_data'})
PROVINCE_INFO_KEYS = frozenset({'province_key', 'province_code', 'name_th', 'name_en'})
PUBLIC_RESULT_SCHEMA = {
    'spatial': {'required': frozenset({'schema', 'catalog_id', 'kind', 'segment_id', 'target_date', 'unit', 'time_block', 'timezone', 'issue_time_local',
                                       'history_cutoff_local', 'latency_days', 'eligibility', 'history_window', 'n_selected_records',
                                       'empty_history_is_not_safety', 'counts_by_year', 'counts_by_month', 'counts_by_time_block', 'categories', 'notes',
                                       'research_status'}),
                'optional': frozenset({'n_same_time_block'}), 'months_key': 'counts_by_month'},
    'province': {'required': frozenset({'schema', 'catalog_id', 'kind', 'province', 'start_month', 'end_month', 'n_months', 'timezone', 'as_of_cutoff',
                                        'months', 'total_records', 'zero_months', 'categories', 'notes', 'research_status'}),
                 'optional': frozenset(), 'months_key': 'months'}}
YEAR_KEY_RE = re.compile(r'[0-9]{4}')


def leaked_keys(obj):
    """Forbidden event-level keys anywhere in a public query result (recursive through dicts and lists)."""
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in FORBIDDEN_OUTPUT_KEYS:
                out.append(k)
            out += leaked_keys(v)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            out += leaked_keys(v)
    return out


def _keyset(obj, required, optional, path, out):
    if type(obj) is not dict:
        out.append(path + ': not an object')
        return False
    miss, extra = sorted(required - set(obj)), sorted(str(k) for k in set(obj) - required - optional)
    if miss:
        out.append(path + ': missing ' + str(miss))
    if extra:
        out.append(path + ': unexpected ' + str(extra)[:200])
    return True


def public_result_violations(res, kind):
    """[] iff res matches the explicit public schema of kind ('spatial' | 'province') exactly and has no forbidden key at any depth."""
    if kind not in PUBLIC_RESULT_SCHEMA:
        raise CatalogError('unknown public result kind ' + repr(kind))
    S, out = PUBLIC_RESULT_SCHEMA[kind], ['forbidden key ' + k for k in leaked_keys(res)]
    if not _keyset(res, S['required'], S['optional'], kind, out):
        return out
    _keyset(res.get('research_status'), RESEARCH_STATUS_KEYS, frozenset(), 'research_status', out)
    cats = res.get('categories')
    if type(cats) is not dict or not set(cats) <= set(CATEGORIES):
        out.append('categories: not an object keyed by ' + str(list(CATEGORIES)))
    else:
        for c, blk in cats.items():
            if _keyset(blk, CATEGORY_BLOCK_KEYS, frozenset(), 'categories.' + c, out):
                tv = blk['top_values'] if 'top_values' in blk else []
                if type(tv) is not list:
                    out.append('categories.' + c + '.top_values: not a list')
                else:
                    for i, t in enumerate(tv):
                        _keyset(t, TOP_VALUE_KEYS, frozenset(), 'categories.' + c + '.top_values[' + str(i) + ']', out)
    ml = res.get(S['months_key'])
    if type(ml) is not list:
        out.append(S['months_key'] + ': not a list')
    else:
        for i, m in enumerate(ml):
            _keyset(m, MONTH_COUNT_KEYS, frozenset(), S['months_key'] + '[' + str(i) + ']', out)
    if type(res.get('notes')) is not list or not all(type(n) is str for n in res['notes']):
        out.append('notes: not a list of strings')
    if kind == 'spatial':
        _keyset(res.get('history_window'), HISTORY_WINDOW_KEYS, frozenset(), 'history_window', out)
        cy, cb = res.get('counts_by_year'), res.get('counts_by_time_block')
        if type(cy) is not dict or not all(YEAR_KEY_RE.fullmatch(str(k)) for k in cy):
            out.append('counts_by_year: not an object keyed by YYYY')
        if type(cb) is not dict or set(cb) != set(TIME_BLOCKS):
            out.append('counts_by_time_block: keys are not exactly ' + str(list(TIME_BLOCKS)))
        if ('n_same_time_block' in res) != (res.get('unit') == '6h'):
            out.append('n_same_time_block present iff unit 6h')
    else:
        _keyset(res.get('province'), PROVINCE_INFO_KEYS, frozenset(), 'province', out)
    return out


def require_public_result(res, kind):
    """Fail closed (internal error, never served) when an answer departs from the explicit public schema."""
    bad = public_result_violations(res, kind)
    if bad:
        raise CatalogError('public ' + kind + ' result violates the public schema: ' + '; '.join(bad[:5]))
    return res


class CatalogQueryError(ValueError):
    def __init__(self, errors):
        super().__init__('; '.join(str(e.get('field')) + ': ' + str(e.get('code')) for e in errors))
        self.errors = errors


def _err(field, code, message, index=None):
    return {'index': index, 'field': field, 'code': code, 'message': message}


def date_seconds(d):
    return (dt.date.fromisoformat(d) - dt.date(1970, 1, 1)).days * 86400


def dates_to_seconds(series):
    return pd.to_datetime(pd.Series(series), format='%Y-%m-%d').to_numpy().astype('datetime64[s]').astype('int64')


def event_end_seconds(dates, blocks, unit):
    """Target end on the Asia/Bangkok wall clock (fixed UTC+7, no DST): day end for daily, block end ('00-06' -> 06:00 ... '18-24' -> next 00:00) for 6h."""
    day = dates_to_seconds(dates)
    if unit == 'daily':
        return day + 86400
    hours = np.array([int(str(b)[3:5]) for b in blocks], dtype='int64')
    return day + hours * 3600


def local_iso(s):
    return (dt.datetime(1970, 1, 1) + dt.timedelta(seconds=int(s))).isoformat() + '+07:00'


def issue_and_cutoff(target_date, qcfg):
    issue = date_seconds(target_date) + int(qcfg['issue_day_offset']) * 86400 + int(qcfg['issue_hour_local']) * 3600
    return issue, issue - int(qcfg['latency_days']) * 86400


def history_eligible(ends, cutoff, window_start):
    """Eligible iff window_start < target_end <= cutoff (equality at the cutoff is eligible; one second later is not)."""
    ends = np.asarray(ends, dtype='int64')
    return (ends <= int(cutoff)) & (ends > int(window_start))


def build_catalog_view(cfg, provinces, prov_monthly, prov_cat, universe_ids, rows, catalog_id):
    """In-memory, read-only query structures from verified tables (used by load_catalog and by the notebook's own controls)."""
    C = cfg['catalog']
    months = month_range(C['first_month'], C['last_month'])
    keys = [p['province_key'] for p in provinces]
    if len(keys) != int(C['expected_provinces']) or len(set(keys)) != len(keys):
        raise CatalogError('catalog provinces are not the expected unique set')
    PM = prov_monthly.copy()
    for c in ['n_records'] + ['known_' + k for k in CATEGORIES] + ['unknown_' + k for k in CATEGORIES]:
        PM[c] = pd.to_numeric(PM[c], errors='raise').astype('int64')
    if len(PM) != len(keys) * len(months) or set(PM['province_key']) != set(keys) or set(PM['year_month']) != set(months):
        raise CatalogError('province monthly table is not the full province x month grid')
    PC = prov_cat.copy()
    PC['n_records'] = pd.to_numeric(PC['n_records'], errors='raise').astype('int64')
    uni = sorted(universe_ids)
    if len(set(uni)) != len(uni):
        raise CatalogError('universe segment ids not unique')
    R = rows.reset_index(drop=True)
    if not R['segment_id'].isin(set(uni)).all() or not R['time_block'].isin(set(TIME_BLOCKS)).all():
        raise CatalogError('selected history rows outside the universe or time blocks')
    idx = {k: np.asarray(v, dtype='int64') for k, v in R.groupby('segment_id').indices.items()}
    empty = np.empty(0, dtype='int64')
    spatial = {'segment_ids': frozenset(uni), 'by_segment': {s: idx.get(s, empty) for s in uni},
               'day_end': event_end_seconds(R['event_date'], R['time_block'], 'daily'),
               'block_end': event_end_seconds(R['event_date'], R['time_block'], '6h'),
               'year': R['event_date'].str[:4].to_numpy(), 'year_month': R['event_date'].str[:7].to_numpy(),
               **{c: R[c].to_numpy(dtype=object) for c in CATEGORIES}, 'n_rows': len(R)}
    return {'catalog_id': catalog_id, 'config': cfg, 'months': months, 'province_set': frozenset(keys),
            'provinces': {p['province_key']: p for p in provinces},
            'prov_monthly': {k: g.set_index('year_month').sort_index() for k, g in PM.groupby('province_key')},
            'prov_cat': {k: g.reset_index(drop=True) for k, g in PC.groupby('province_key')},
            'spatial': spatial, 'data_start_s': date_seconds(months[0] + '-01'),
            'data_end_s': date_seconds(next_month(months[-1]) + '-01')}


def next_month(ym):
    y, m = int(ym[:4]), int(ym[5:7])
    return '%04d-%02d' % ((y + 1, 1) if m == 12 else (y, m + 1))


def _verify_output_files(base, files, required, label):
    base = Path(base)
    if not isinstance(files, dict) or not set(required) <= set(files):
        raise CatalogError(label + ': manifest output list lacks required files ' + str(sorted(set(required) - set(files or {}))))
    for rel, sha in sorted(files.items()):
        pp = PurePosixPath(rel)
        if pp.is_absolute() or any(p in ('..', '.', '') for p in pp.parts):
            raise CatalogError(label + ': unsafe output path in manifest ' + repr(rel))
        p = base.joinpath(*pp.parts)
        if symlinked_components(p, base):
            raise CatalogError(label + ': symlinked output path ' + rel)
        verify_pinned(p, sha, label + ' ' + rel)
    return True


def load_catalog(report_dir, data_dir, config_path, expected_manifest_sha256, expected_functions_sha256, expected_config_sha256=None,
                 source_notebook=None):
    """Trusted, server-configured paths only (never client supplied). Order: manifest sha == pin -> strict manifest contract (complete,
    research-only, every check pass, FUNCTIONS sha == pin) -> config sha -> optional source notebook FUNCTIONS/code-cell hashes -> every
    report/data file sha -> load and validate tables. Returns the read-only catalog view."""
    report_dir, data_dir = Path(report_dir), Path(data_dir)
    mp = report_dir / 'catalog_manifest.json'
    msha = verify_pinned(mp, expected_manifest_sha256, 'catalog manifest (runtime pin)')
    man = read_strict_json(mp)
    if man.get('schema') != MANIFEST_SCHEMA or man.get('producer') != CATALOG_PRODUCER or man.get('status') != 'complete':
        raise CatalogError('catalog manifest is not a complete notebook-14 catalog')
    if man.get('production_ready') is not False or man.get('research_only') is not True:
        raise CatalogError('catalog manifest must stay research-only (production_ready false)')
    if not man.get('checks') or any(c.get('status') != 'pass' for c in man['checks']):
        raise CatalogError('catalog manifest records a failed or missing check')
    if not expected_functions_sha256 or man.get('functions_cells_sha256') != expected_functions_sha256:
        raise CatalogError('catalog was produced by other notebook-14 FUNCTIONS than the pinned ones')
    csha = verify_pinned(config_path, man.get('config_sha256'), 'catalog config')
    if expected_config_sha256 is not None and csha != expected_config_sha256:
        raise CatalogError('catalog config sha differs from the expected pin')
    if source_notebook is not None:
        cells = notebook_code_cells(source_notebook)
        if functions_sha(cells) != expected_functions_sha256 or code_cells_sha(cells) != man.get('code_cells_sha256'):
            raise CatalogError('source notebook FUNCTIONS/code cells differ from the catalog manifest')
    outs = man.get('outputs') or {}
    _verify_output_files(report_dir, outs.get('report_files'), REPORT_FILES_REQUIRED, 'catalog report')
    _verify_output_files(data_dir, outs.get('data_files'), DATA_FILES_REQUIRED, 'catalog data')
    cfg = read_strict_json(config_path)
    validate_config(cfg)
    prov = read_strict_json(data_dir / 'provinces.json')['provinces']
    uni = read_str_csv(data_dir / 'universe_segment_ids.csv')['segment_id'].tolist()
    rows = read_str_csv(data_dir / 'selected_history_rows.csv.gz')
    view = build_catalog_view(cfg, prov, read_str_csv(report_dir / 'province_monthly.csv'), read_str_csv(report_dir / 'province_category_monthly.csv'),
                              uni, rows, msha[:16])
    view.update(manifest=man, manifest_sha256=msha, loaded_utc=utcnow())
    return view


# ---------- strict request parsing ----------
def parse_request_json(text, max_bytes=65536):
    """Strict JSON body: bounded size, UTF-8, no duplicate keys, no NaN/Infinity constants."""
    if isinstance(text, (bytes, bytearray)):
        if len(text) > int(max_bytes):
            raise CatalogQueryError([_err(None, 'request_too_large', 'at most ' + str(max_bytes) + ' bytes')])
        try:
            text = bytes(text).decode('utf-8')
        except UnicodeDecodeError:
            raise CatalogQueryError([_err(None, 'invalid_encoding', 'request must be UTF-8 JSON')])
    if type(text) is not str:
        raise CatalogQueryError([_err(None, 'invalid_type', 'request must be JSON text')])
    if len(text.encode('utf-8')) > int(max_bytes):
        raise CatalogQueryError([_err(None, 'request_too_large', 'at most ' + str(max_bytes) + ' bytes')])
    try:
        return strict_json_loads(text)
    except (CatalogError, ValueError) as e:
        raise CatalogQueryError([_err(None, 'invalid_json', str(e)[:200])])


def _fields(req, allowed, required, errs, index=None):
    if type(req) is not dict:
        errs.append(_err(None, 'invalid_type', 'request must be a JSON object', index))
        return {}
    for k in req:
        if type(k) is not str or k not in allowed:
            errs.append(_err(str(k)[:40], 'unknown_field', 'allowed fields: ' + str(list(allowed)), index))
    for k in required:
        if k not in req:
            errs.append(_err(k, 'missing_field', k + ' is required', index))
    return {k: v for k, v in req.items() if type(k) is str and k in allowed}


def _str_field(q, k, errs, index):
    v = q.get(k)
    if v is None and k not in q:
        return None
    if type(v) is not str or not v or len(v) > MAX_STR:
        errs.append(_err(k, 'invalid_type', k + ' must be a non-empty string', index))
        return None
    return v


def _int_field(q, k, lo, hi, default, errs, index):
    if k not in q:
        return default
    v = q[k]
    if type(v) is not int:  # bool, float and strings refused
        errs.append(_err(k, 'invalid_type', k + ' must be an integer', index))
        return default
    if not lo <= v <= hi:
        errs.append(_err(k, 'out_of_range', k + ' must be in [' + str(lo) + ', ' + str(hi) + ']', index))
    return v


def _categories_field(q, errs, index):
    if 'categories' not in q:
        return list(CATEGORIES)
    v = q['categories']
    if type(v) is not list or not v or any(type(c) is not str for c in v) or len(set(v)) != len(v) or not set(v) <= set(CATEGORIES):
        errs.append(_err('categories', 'invalid_category', 'non-empty list without duplicates from ' + str(list(CATEGORIES)), index))
        return list(CATEGORIES)
    return [c for c in CATEGORIES if c in v]


def _category_block(counts, total, top_k):
    """counts: {value: n} with '' = unknown. Known + unknown must equal the record total (else internal error)."""
    unknown = int(counts.get('', 0))
    known = sorted(((str(v), int(n)) for v, n in counts.items() if v != '' and int(n) > 0), key=lambda t: (-t[1], t[0]))
    k = sum(n for _, n in known)
    if k + unknown != int(total):
        raise CatalogError('category marginal ' + str(k + unknown) + ' != record total ' + str(total))
    top = known[:top_k]
    return {'denominator_records': int(total), 'known_records': k, 'unknown_records': unknown,
            'known_share': (k / total) if total else None, 'distinct_known_values': len(known),
            'top_values': [{'value': v, 'records': n} for v, n in top], 'other_known_records': k - sum(n for _, n in top)}


# ---------- province retrospective history ----------
def parse_province_request(cat, req, index=None):
    Q, errs = cat['config']['query'], []
    q = _fields(req, PROVINCE_KEYS, PROVINCE_REQUIRED, errs, index)
    pk = _str_field(q, 'province_key', errs, index)
    if pk is not None and pk not in cat['province_set']:
        errs.append(_err('province_key', 'unknown_province', 'not one of the ' + str(len(cat['province_set'])) + ' catalog province keys', index))
    first, last = cat['months'][0], cat['months'][-1]
    ms = {}
    for f in ('start_month', 'end_month'):
        v = _str_field(q, f, errs, index)
        if v is None:
            continue
        if not MONTH_RE.fullmatch(v):
            errs.append(_err(f, 'invalid_format', 'expected YYYY-MM (CE)', index))
        elif not first <= v <= last:
            errs.append(_err(f, 'out_of_catalog_range', 'catalog covers ' + first + '..' + last, index))
        else:
            ms[f] = v
    if len(ms) == 2:
        if ms['start_month'] > ms['end_month']:
            errs.append(_err('end_month', 'invalid_range', 'start_month must not be after end_month', index))
        elif len(month_range(ms['start_month'], ms['end_month'])) > int(Q['max_months']):
            errs.append(_err('end_month', 'range_too_large', 'at most ' + str(Q['max_months']) + ' months', index))
    cats = _categories_field(q, errs, index)
    top_k = _int_field(q, 'top_k', 1, int(Q['max_top_k']), int(Q['default_top_k']), errs, index)
    if errs:
        raise CatalogQueryError(errs)
    return {'province_key': pk, 'start_month': ms['start_month'], 'end_month': ms['end_month'], 'categories': cats, 'top_k': top_k}


def query_province_history(cat, req):
    """PUBLIC. Retrospective province x month recorded counts (explicit zeros) + category marginals with known/unknown denominators."""
    q = parse_province_request(cat, req)
    months = month_range(q['start_month'], q['end_month'])
    m = cat['prov_monthly'][q['province_key']].loc[months]
    total = int(m['n_records'].sum())
    C = cat['prov_cat'].get(q['province_key'])
    C = C.iloc[0:0] if C is None else C[C['year_month'].isin(set(months))]
    cats = {}
    for c in q['categories']:
        cats[c] = _category_block(C[C['category'] == c].groupby('value')['n_records'].sum().to_dict(), total, q['top_k'])
        if cats[c]['unknown_records'] != int(m['unknown_' + c].sum()):
            raise CatalogError('province unknown marginal differs from the grid for ' + c)
    p = cat['provinces'][q['province_key']]
    return require_public_result(jsonable({'schema': PROVINCE_SCHEMA, 'catalog_id': cat['catalog_id'], 'kind': 'province_retrospective_history',
                     'province': {'province_key': p['province_key'], 'province_code': p['province_code'], 'name_th': p['name_th'], 'name_en': p['name_en']},
                     'start_month': q['start_month'], 'end_month': q['end_month'], 'n_months': len(months), 'timezone': 'Asia/Bangkok',
                     'as_of_cutoff': None, 'months': [{'year_month': ym, 'records': int(n)} for ym, n in m['n_records'].items()],
                     'total_records': total, 'zero_months': int((m['n_records'] == 0).sum()), 'categories': cats,
                     'notes': [PROVINCE_NOTE, CATEGORY_NOTE], 'research_status': RESEARCH_STATUS}), 'province')


# ---------- spatial as-of history ----------
def parse_spatial_request(cat, req, index=None):
    Q, errs = cat['config']['query'], []
    q = _fields(req, SPATIAL_KEYS, SPATIAL_REQUIRED, errs, index)
    seg = _str_field(q, 'segment_id', errs, index)
    if seg is not None and seg not in cat['spatial']['segment_ids']:
        errs.append(_err('segment_id', 'unknown_segment', 'not a supported notebook-19 universe segment id', index))
    td = _str_field(q, 'target_date', errs, index)
    if td is not None:
        try:
            ok = bool(DATE_RE.fullmatch(td)) and dt.date.fromisoformat(td).isoformat() == td
        except ValueError:
            ok = False
        if not ok:
            errs.append(_err('target_date', 'invalid_date', 'expected an exact ISO date YYYY-MM-DD', index))
            td = None
        elif not Q['min_target_date'] <= td <= Q['max_target_date']:
            errs.append(_err('target_date', 'out_of_range', 'target_date must be in ' + Q['min_target_date'] + '..' + Q['max_target_date'], index))
    unit = _str_field(q, 'unit', errs, index)
    if unit is not None and unit not in UNITS:
        errs.append(_err('unit', 'invalid_unit', 'unit must be one of ' + str(list(UNITS)), index))
    blk = None
    if unit == '6h':
        if 'time_block' not in q:
            errs.append(_err('time_block', 'missing_field', 'time_block is required for unit 6h', index))
        else:
            blk = _str_field(q, 'time_block', errs, index)
            if blk is not None and blk not in TIME_BLOCKS:
                errs.append(_err('time_block', 'invalid_time_block', 'time_block must be one of ' + str(list(TIME_BLOCKS)), index))
    elif 'time_block' in q:
        errs.append(_err('time_block', 'not_allowed', 'time_block is only allowed for unit 6h', index))
    lb = _int_field(q, 'lookback_days', 1, int(Q['max_lookback_days']), int(Q['default_lookback_days']), errs, index)
    cats = _categories_field(q, errs, index)
    top_k = _int_field(q, 'top_k', 1, int(Q['max_top_k']), int(Q['default_top_k']), errs, index)
    if errs:
        raise CatalogQueryError(errs)
    return {'segment_id': seg, 'target_date': td, 'unit': unit, 'time_block': blk, 'lookback_days': lb, 'categories': cats, 'top_k': top_k}


def spatial_history_answer(cat, q):
    """Aggregate answer for a parsed request (internal; parse first). Internal guards: no included record ends after the cutoff and all
    marginals reconcile to N."""
    S, Q = cat['spatial'], cat['config']['query']
    idx = S['by_segment'][q['segment_id']]
    issue, cutoff = issue_and_cutoff(q['target_date'], Q)
    ws = cutoff - int(q['lookback_days']) * 86400
    ends = S['day_end'] if q['unit'] == 'daily' else S['block_end']
    sel = idx[history_eligible(ends[idx], cutoff, ws)]
    if len(sel) and int(ends[sel].max()) > cutoff:
        raise CatalogError('internal leak guard: a record after the cutoff was selected')
    n = len(sel)
    es, ee = max(ws, cat['data_start_s']), min(cutoff, cat['data_end_s'])
    months = month_range(local_iso(es)[:7], local_iso(ee - 1)[:7]) if ee > es else []
    ym = pd.Series(S['year_month'][sel]).value_counts().to_dict()
    if sum(int(ym.get(m, 0)) for m in months) != n:
        raise CatalogError('spatial month counts do not reconcile to N')
    yr = pd.Series(S['year'][sel]).value_counts().to_dict()
    blocks = pd.Series(S['time_block'][sel]).value_counts().to_dict()
    cats = {c: _category_block(pd.Series(S[c][sel], dtype=object).value_counts().to_dict(), n, q['top_k']) for c in q['categories']}
    out = {'schema': SPATIAL_SCHEMA, 'catalog_id': cat['catalog_id'], 'kind': 'spatial_selected_history_as_of', 'segment_id': q['segment_id'],
           'target_date': q['target_date'], 'unit': q['unit'], 'time_block': q['time_block'], 'timezone': 'Asia/Bangkok',
           'issue_time_local': local_iso(issue), 'history_cutoff_local': local_iso(cutoff), 'latency_days': int(Q['latency_days']),
           'eligibility': ('target day end <= cutoff' if q['unit'] == 'daily' else 'target 6 h block end <= cutoff'),
           'history_window': {'lookback_days': int(q['lookback_days']), 'window_start_local_exclusive': local_iso(ws), 'window_end_local_inclusive': local_iso(cutoff),
                              'catalog_data_start_local': local_iso(cat['data_start_s']), 'catalog_data_end_local_exclusive': local_iso(cat['data_end_s']),
                              'covered_days': round(max(0, ee - es) / 86400.0, 4),
                              'window_fully_inside_catalog_data': bool(ws >= cat['data_start_s'] and cutoff <= cat['data_end_s'])},
           'n_selected_records': n, 'empty_history_is_not_safety': True,
           'counts_by_year': {y: int(yr[y]) for y in sorted(yr)},
           'counts_by_month': [{'year_month': m, 'records': int(ym.get(m, 0))} for m in months],
           'counts_by_time_block': {b: int(blocks.get(b, 0)) for b in TIME_BLOCKS},
           'categories': cats, 'notes': [SPATIAL_NOTE, CATEGORY_NOTE], 'research_status': RESEARCH_STATUS}
    if q['unit'] == '6h':
        out['n_same_time_block'] = int(blocks.get(q['time_block'], 0))
    if sum(out['counts_by_time_block'].values()) != n or sum(out['counts_by_year'].values()) != n:
        raise CatalogError('spatial marginals do not reconcile to N')
    return require_public_result(jsonable(out), 'spatial')


def query_spatial_history(cat, req):
    """PUBLIC. One strict request {segment_id, target_date, unit[, time_block][, lookback_days][, categories][, top_k]}."""
    return spatial_history_answer(cat, parse_spatial_request(cat, req))


def query_spatial_history_batch(cat, requests):
    """PUBLIC. 1..max_batch strict requests (no duplicates); all are validated before any answer; errors carry the request index."""
    mx = int(cat['config']['query']['max_batch'])
    if type(requests) is not list or not requests:
        raise CatalogQueryError([_err(None, 'invalid_type', 'requests must be a non-empty JSON array')])
    if len(requests) > mx:
        raise CatalogQueryError([_err(None, 'too_many_requests', 'at most ' + str(mx) + ' requests per batch')])
    errs, parsed, seen = [], [], set()
    for i, r in enumerate(requests):
        try:
            p = parse_spatial_request(cat, r, index=i)
        except CatalogQueryError as e:
            errs.extend(e.errors)
            continue
        key = canonical_sha(p)
        if key in seen:
            errs.append(_err(None, 'duplicate_request', 'identical request already in this batch', i))
        seen.add(key)
        parsed.append(p)
    if errs:
        raise CatalogQueryError(errs)
    return {'schema': SPATIAL_SCHEMA + '+batch', 'catalog_id': cat['catalog_id'], 'results': [spatial_history_answer(cat, p) for p in parsed]}


def catalog_summary(cat):
    """JSON-safe catalog metadata for a metadata endpoint (no filesystem paths, no event rows)."""
    man, Q = cat.get('manifest') or {}, cat['config']['query']
    return jsonable({'schema': MANIFEST_SCHEMA, 'catalog_id': cat['catalog_id'], 'catalog_manifest_sha256': cat.get('manifest_sha256'),
                     'functions_cells_sha256': man.get('functions_cells_sha256'), 'first_month': cat['months'][0], 'last_month': cat['months'][-1],
                     'n_provinces': len(cat['province_set']), 'n_universe_segments': len(cat['spatial']['segment_ids']),
                     'n_selected_records': cat['spatial']['n_rows'], 'categories': list(CATEGORIES), 'time_blocks': list(TIME_BLOCKS),
                     'province_query': {'fields': list(PROVINCE_KEYS), 'required': list(PROVINCE_REQUIRED), 'max_months': Q['max_months'], 'max_top_k': Q['max_top_k']},
                     'spatial_query': {'fields': list(SPATIAL_KEYS), 'required': list(SPATIAL_REQUIRED), 'units': list(UNITS),
                                       'target_date_range': [Q['min_target_date'], Q['max_target_date']], 'max_lookback_days': Q['max_lookback_days'],
                                       'max_batch': Q['max_batch'], 'max_top_k': Q['max_top_k'], 'issue_time': 'target_date - 1 day 18:00 Asia/Bangkok',
                                       'latency_days': Q['latency_days']},
                     'notes': [PROVINCE_NOTE, SPATIAL_NOTE, CATEGORY_NOTE], 'research_status': RESEARCH_STATUS})


def list_provinces(cat):
    """Province keys/names with the REPRESENTATIVE label point (never an event location)."""
    return jsonable({'catalog_id': cat['catalog_id'], 'provinces': [
        {'province_key': p['province_key'], 'province_code': p['province_code'], 'name_th': p['name_th'], 'name_en': p['name_en'],
         'representative_point': {'lat': p['lat'], 'lon': p['lon'], 'meaning': p['point_meaning']}} for p in cat['provinces'].values()]})


In [ ]:
# FUNCTIONS (5/5): independent brute-force / negative query controls, figures, manifest and Thai report builders. The controls re-derive the
# as-of rule with plain datetime arithmetic (not the vectorised query path) and inject synthetic future records to prove they never count.
# Privacy controls check every public answer against the explicit public schema (FUNCTIONS 4), forbidden keys at any depth and EXACT
# membership in the actual retained event-UID set. A UID-shaped regex is not used: stable public segment ids legitimately contain
# '...:doh:<route>:p0:...' and DOH route numbers 25xx look like a BE year ('2577:p0'), which is not a leak.
def uid_index(uids):
    """Detector index over the ACTUAL retained event UIDs (event_uid == '<BE year>:<ACC_CODE>')."""
    U = frozenset(str(u) for u in uids)
    if not U:
        raise CatalogError('empty retained UID set: the UID leak detector would be vacuous')
    if not all(':' in u and u.split(':', 1)[0] and u.split(':', 1)[1] for u in U):
        raise CatalogError('retained UIDs are not all <BE year>:<ACC_CODE>')
    prefixes = sorted({u.split(':', 1)[0] + ':' for u in U})
    start = re.compile('(?<![0-9])(?:' + '|'.join(re.escape(p) for p in prefixes) + ')')
    return {'uids': U, 'lengths': sorted({len(u) for u in U}), 'start': start, 'n_uids': len(U), 'prefixes': prefixes}


def _strings(obj):
    if isinstance(obj, str):
        yield obj
    elif isinstance(obj, dict):
        for k, v in obj.items():
            yield str(k)
            yield from _strings(v)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            yield from _strings(v)


def retained_uid_hits(obj, index):
    """Retained UIDs found in obj (any key or string value, any depth): a whole string equal to a UID, or a UID embedded as a token (not
    preceded by a digit, not followed by a letter/digit). Callers record len(hits) only, never the hit strings."""
    U, L, start = index['uids'], index['lengths'], index['start']
    hits = []
    for s in _strings(obj):
        if s in U:
            hits.append(s)
            continue
        for m in start.finditer(s):
            tail = s[m.start():]
            for n in L:
                if n > len(tail):
                    break
                if tail[:n] in U and (n == len(tail) or not tail[n].isalnum()):
                    hits.append(tail[:n])
                    break
    return hits


def audit_public_result(res, kind, index):
    """Aggregate privacy audit of one public answer: schema violations, forbidden keys (recursive), retained-UID hits. Counts only."""
    return {'schema_violations': len(public_result_violations(res, kind)), 'forbidden_keys': len(leaked_keys(res)),
            'retained_uid_hits': len(retained_uid_hits(res, index))}


def _brute_end(date_s, block, unit):
    d = dt.datetime.fromisoformat(date_s)
    return d + (dt.timedelta(days=1) if unit == 'daily' else dt.timedelta(hours=int(block[3:5])))


def brute_force_n(R_seg, target_date, unit, lookback_days, qcfg):
    """Independent re-derivation of N on raw row context with datetime objects."""
    issue = dt.datetime.fromisoformat(target_date) + dt.timedelta(days=int(qcfg['issue_day_offset']), hours=int(qcfg['issue_hour_local']))
    cutoff = issue - dt.timedelta(days=int(qcfg['latency_days']))
    start = cutoff - dt.timedelta(days=int(lookback_days))
    n = 0
    for d, b in zip(R_seg['event_date'], R_seg['time_block']):
        e = _brute_end(d, b, unit)
        if start < e <= cutoff:
            n += 1
    return n, cutoff


def _expect_error(fn, code):
    try:
        fn()
    except CatalogQueryError as e:
        codes = [x['code'] for x in e.errors]
        return code in codes, ','.join(codes)
    except Exception as e:  # any other exception is a control failure, reported
        return False, type(e).__name__ + ': ' + str(e)[:120]
    return False, 'accepted'


def run_query_controls(view, R, cfg, retained_uids):
    """Rows (control, expected, observed, ok). R = the selected row context the view was built from; retained_uids = the actual accepted
    retained event UIDs (the privacy audit matches them exactly). Every public answer produced here is audited; observed values are counts."""
    Q, rng = cfg['query'], np.random.default_rng(int(cfg['controls']['seed']))
    U = uid_index(retained_uids)
    rows, audit = [], {'spatial': [0, 0, 0, 0], 'province': [0, 0, 0, 0]}  # answers, schema violations, forbidden keys, retained-UID hits

    def add(name, expected, observed, ok):
        rows.append({'control': name, 'expected': str(expected)[:300], 'observed': str(observed)[:300], 'ok': bool(ok)})

    def seen(res, kind):
        a, t = audit_public_result(res, kind, U), audit[kind]
        t[0] += 1
        t[1] += a['schema_violations']
        t[2] += a['forbidden_keys']
        t[3] += a['retained_uid_hits']
        return res

    by_seg = {k: g for k, g in R.groupby('segment_id')}
    segs = sorted(by_seg)
    lo, hi = dt.date.fromisoformat(Q['min_target_date']), dt.date.fromisoformat(Q['max_target_date'])
    bad = 0
    for i in range(int(cfg['controls']['n_bruteforce_queries'])):
        s = segs[int(rng.integers(len(segs)))]
        td = (lo + dt.timedelta(days=int(rng.integers((hi - lo).days + 1)))).isoformat()
        unit = UNITS[int(rng.integers(2))]
        lb = int(rng.integers(1, int(Q['max_lookback_days']) + 1)) if i % 2 else int(Q['default_lookback_days'])
        req = {'segment_id': s, 'target_date': td, 'unit': unit, 'lookback_days': lb}
        if unit == '6h':
            req['time_block'] = TIME_BLOCKS[int(rng.integers(4))]
        res = seen(query_spatial_history(view, req), 'spatial')
        n, _ = brute_force_n(by_seg[s], td, unit, lb, Q)
        bad += int(res['n_selected_records'] != n)
    add('brute-force N == query N (random segments/dates/units/lookbacks)', 0, bad, bad == 0)

    # privacy detector counterexamples on copies of a real answer: a REAL retained UID planted anywhere and forbidden nested keys must be
    # flagged; all public universe segment ids (incl. DOH 25xx route tokens) and the ISO dates/months must not be
    base = seen(query_spatial_history(view, {'segment_id': segs[0], 'target_date': '2024-06-01', 'unit': 'daily'}), 'spatial')
    real_uid, c0 = min(U['uids']), CATEGORIES[0]
    planted = [('real retained UID as a nested category value', lambda d: d['categories'][c0]['top_values'].append({'value': real_uid, 'records': 1}), 'retained_uid_hits'),
               ('real retained UID embedded in note text', lambda d: d['notes'].append('record ' + real_uid + ' near km 3'), 'retained_uid_hits'),
               ('real retained UID as an object key', lambda d: d['counts_by_year'].update({real_uid: 1}), 'retained_uid_hits'),
               ('nested event_uid key', lambda d: d['history_window'].update(detail=[{'event_uid': 'x'}]), 'forbidden_keys'),
               ('nested source_row key', lambda d: d['categories'][c0]['top_values'].append({'value': 'a', 'records': 1, 'source_row': 7}), 'forbidden_keys'),
               ('nested individual coordinates', lambda d: d['history_window'].update(where={'lat': 13.7, 'lon': 100.5}), 'forbidden_keys'),
               ('extra top-level narrative field', lambda d: d.update(narrative_text='x'), 'schema_violations')]
    for name, mutate, key in planted:
        d = json.loads(json.dumps(base))
        mutate(d)
        got = audit_public_result(d, 'spatial', U)[key]
        add('detector flags planted ' + name, '>=1 ' + key, got, got >= 1)
    pub = sorted(view['spatial']['segment_ids'])
    shaped = sum(1 for s in pub if re.search(r'(?<![0-9])25[0-9]{2}:', s))
    fp = len(retained_uid_hits({'segment_ids': pub, 'months': view['months'], 'times': [base['issue_time_local'], base['history_cutoff_local']]}, U))
    add('detector tolerates all ' + str(len(pub)) + ' public universe segment ids (' + str(shaped) + ' carry a 25xx: route token) and ISO dates', 0, fp, fp == 0)

    # boundary: a 12-18 record ends at 18:00 = cutoff when target = event_date + 31 days (equality eligible); target one day earlier excludes it
    cand = R[(R['time_block'] == '12-18') & (R['event_date'] >= '2021-03-01')]
    r = cand.iloc[0]
    d0 = dt.date.fromisoformat(r['event_date'])
    for unit, delta, should in (('6h', 31, True), ('6h', 30, False), ('daily', 32, True), ('daily', 31, False)):
        td = (d0 + dt.timedelta(days=delta)).isoformat()
        _, cutoff = brute_force_n(by_seg[r['segment_id']], td, unit, Q['default_lookback_days'], Q)
        end = _brute_end(r['event_date'], r['time_block'], unit)
        add('boundary ' + unit + ' target=event+' + str(delta) + 'd (end ' + end.isoformat() + ' vs cutoff ' + cutoff.isoformat() + ')',
            'eligible' if should else 'excluded', 'eligible' if end <= cutoff else 'excluded', (end <= cutoff) == should)
        req = {'segment_id': r['segment_id'], 'target_date': td, 'unit': unit, **({'time_block': '12-18'} if unit == '6h' else {})}
        n, _ = brute_force_n(by_seg[r['segment_id']], td, unit, Q['default_lookback_days'], Q)
        got = seen(query_spatial_history(view, req), 'spatial')['n_selected_records']
        add('boundary query N == brute force (' + unit + ', +' + str(delta) + 'd)', n, got, n == got)
    e0 = int(event_end_seconds([r['event_date']], [r['time_block']], '6h')[0])
    add('history_eligible: end == cutoff included', True, bool(history_eligible([e0], e0, e0 - 86400)[0]), bool(history_eligible([e0], e0, e0 - 86400)[0]))
    add('history_eligible: end one second after cutoff excluded', False, bool(history_eligible([e0], e0 - 1, e0 - 86400)[0]),
        not bool(history_eligible([e0], e0 - 1, e0 - 86400)[0]))

    # future-leak negative control: inject records at/after the cutoff into a copy; N must not change
    s, td = r['segment_id'], (d0 + dt.timedelta(days=200)).isoformat()
    tdd = dt.date.fromisoformat(td)
    fut = [(tdd - dt.timedelta(days=30)).isoformat(), (tdd - dt.timedelta(days=1)).isoformat(), td, (tdd + dt.timedelta(days=10)).isoformat()]
    inj = pd.DataFrame([{'segment_id': s, 'event_date': f, 'time_block': b, 'ce_year': int(f[:4]), 'accident_type': 'INJECTED_FUTURE',
                         'presumed_cause': 'INJECTED_FUTURE', 'weather': 'INJECTED_FUTURE'} for f in fut for b in TIME_BLOCKS])
    R2 = pd.concat([R, inj], ignore_index=True).sort_values(['segment_id', 'event_date', 'time_block'], kind='stable').reset_index(drop=True)
    v2 = build_catalog_view(cfg, list(view['provinces'].values()), _grid_frame(view), _cat_frame(view), sorted(view['spatial']['segment_ids']), R2, 'control')
    for unit in UNITS:
        req = {'segment_id': s, 'target_date': td, 'unit': unit, **({'time_block': '18-24'} if unit == '6h' else {})}
        a, b = seen(query_spatial_history(view, req), 'spatial'), seen(query_spatial_history(v2, req), 'spatial')
        txt = json.dumps(b, ensure_ascii=False)
        add('injected future records never counted (' + unit + ')', a['n_selected_records'], b['n_selected_records'],
            a['n_selected_records'] == b['n_selected_records'] and 'INJECTED_FUTURE' not in txt)

    # empty-history segment
    empty = sorted(view['spatial']['segment_ids'] - set(segs))
    if empty:
        res = seen(query_spatial_history(view, {'segment_id': empty[0], 'target_date': '2025-06-01', 'unit': 'daily'}), 'spatial')
        add('empty-history segment returns N=0 with not-safety flag', 'N=0 flag', str(res['n_selected_records']) + ' ' + str(res['empty_history_is_not_safety']),
            res['n_selected_records'] == 0 and res['empty_history_is_not_safety'] is True)
    good = {'segment_id': segs[0], 'target_date': '2024-06-01', 'unit': 'daily'}
    invalid = [('unknown segment id', dict(good, segment_id='not-a-segment'), 'unknown_segment'),
               ('bool top_k', dict(good, top_k=True), 'invalid_type'),
               ('float lookback_days', dict(good, lookback_days=30.0), 'invalid_type'),
               ('extra field', dict(good, event_uid='x'), 'unknown_field'),
               ('missing unit', {k: v for k, v in good.items() if k != 'unit'}, 'missing_field'),
               ('target date 2028', dict(good, target_date='2028-01-01'), 'out_of_range'),
               ('target date 2020', dict(good, target_date='2020-12-31'), 'out_of_range'),
               ('invalid calendar date', dict(good, target_date='2024-02-30'), 'invalid_date'),
               ('non-ISO date', dict(good, target_date='2024-6-1'), 'invalid_date'),
               ('6h without time_block', dict(good, unit='6h'), 'missing_field'),
               ('daily with time_block', dict(good, time_block='00-06'), 'not_allowed'),
               ('invalid time block', dict(good, unit='6h', time_block='06-11'), 'invalid_time_block'),
               ('lookback beyond 5 years', dict(good, lookback_days=int(Q['max_lookback_days']) + 1), 'out_of_range'),
               ('top_k above 10', dict(good, top_k=int(Q['max_top_k']) + 1), 'out_of_range'),
               ('unknown category', dict(good, categories=['vehicle_1']), 'invalid_category'),
               ('duplicate category', dict(good, categories=['weather', 'weather']), 'invalid_category'),
               ('non-object request', ['segment_id'], 'invalid_type')]
    for name, req, code in invalid:
        ok, obs = _expect_error(lambda req=req: query_spatial_history(view, req), code)
        add('reject ' + name, code, obs, ok)
    for name, reqs, code in (('batch of max_batch+1', [dict(good, lookback_days=i + 1) for i in range(int(Q['max_batch']) + 1)], 'too_many_requests'),
                             ('duplicate batch requests', [good, dict(good)], 'duplicate_request'), ('empty batch', [], 'invalid_type')):
        ok, obs = _expect_error(lambda reqs=reqs: query_spatial_history_batch(view, reqs), code)
        add('reject ' + name, code, obs, ok)
    okb = query_spatial_history_batch(view, [good, dict(good, unit='6h', time_block='00-06')])
    for ans in okb['results']:
        seen(ans, 'spatial')
    add('batch of 2 answers 2', 2, len(okb['results']), len(okb['results']) == 2)
    for name, txt, code in (('NaN JSON', '{"segment_id": "a", "lookback_days": NaN}', 'invalid_json'),
                            ('duplicate JSON keys', '{"unit": "daily", "unit": "6h"}', 'invalid_json'),
                            ('oversized JSON', '{"a": "' + 'x' * int(Q['max_request_bytes']) + '"}', 'request_too_large')):
        ok, obs = _expect_error(lambda txt=txt: parse_request_json(txt, Q['max_request_bytes']), code)
        add('reject ' + name, code, obs, ok)

    # province controls
    pk = sorted(view['province_set'])[0]
    pgood = {'province_key': pk, 'start_month': view['months'][0], 'end_month': view['months'][-1]}
    tot = sum(seen(query_province_history(view, dict(pgood, province_key=p, categories=['time_block'])), 'province')['total_records']
              for p in sorted(view['province_set']))
    grid = int(sum(int(g['n_records'].sum()) for g in view['prov_monthly'].values()))
    add('sum of ' + str(len(view['province_set'])) + ' full-range province queries == grid total', grid, tot, tot == grid)
    pres = seen(query_province_history(view, pgood), 'province')
    add('province result not as-of', None, pres['as_of_cutoff'], pres['as_of_cutoff'] is None)
    pleak = json.loads(json.dumps(pres))
    pleak['months'][0]['event_uids'] = [real_uid]
    got = audit_public_result(pleak, 'province', U)
    add('detector flags planted nested event_uids list in a province result', '>=1 forbidden key and >=1 retained UID',
        str(got['forbidden_keys']) + ' ' + str(got['retained_uid_hits']), got['forbidden_keys'] >= 1 and got['retained_uid_hits'] >= 1)
    for name, req, code in (('unknown province', dict(pgood, province_key='Atlantis'), 'unknown_province'),
                            ('month after catalog', dict(pgood, end_month=next_month(view['months'][-1])), 'out_of_catalog_range'),
                            ('month 13', dict(pgood, end_month='2024-13'), 'invalid_format'),
                            ('start after end', dict(pgood, start_month=view['months'][-1], end_month=view['months'][0]), 'invalid_range'),
                            ('extra field', dict(pgood, segment_id='x'), 'unknown_field'),
                            ('int month', dict(pgood, start_month=202101), 'invalid_type')):
        ok, obs = _expect_error(lambda req=req: query_province_history(view, req), code)
        add('reject province ' + name, code, obs, ok)

    # privacy audit over EVERY public answer produced above (aggregate counts only; hit strings are never recorded)
    for kind in ('spatial', 'province'):
        n, sv, fk, uh = audit[kind]
        add('all ' + str(n) + ' ' + kind + ' answers match the explicit public schema', 0, sv, n > 0 and sv == 0)
        add('no forbidden event-level key at any depth in ' + str(n) + ' ' + kind + ' answers', 0, fk, n > 0 and fk == 0)
        add('no retained event UID (exact match against ' + str(U['n_uids']) + ' retained UIDs) in ' + str(n) + ' ' + kind + ' answers', 0, uh,
            n > 0 and uh == 0)
    return pd.DataFrame(rows)


def _grid_frame(view):
    return pd.concat([g.reset_index().assign(province_key=k) for k, g in view['prov_monthly'].items()], ignore_index=True)


def _cat_frame(view):
    return pd.concat(list(view['prov_cat'].values()), ignore_index=True)


# ---------- figures (descriptive coverage / history distribution; NOT model risk or forecasts) ----------
def make_figures(fig_dir, cov, by_year, spatial_monthly, seg):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig_dir = Path(fig_dir)
    fig_dir.mkdir(parents=True, exist_ok=True)
    out = {}
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
    x = np.arange(len(cov))
    ax[0].bar(x - 0.2, cov['unknown_before_recovery'], 0.4, label='unknown before (legacy V1 descriptors only)', color='#bbbbbb')
    ax[0].bar(x + 0.2, cov['unknown_after_recovery'], 0.4, label='unknown after exact raw recovery', color='#3b6ea5')
    for i, (a, b) in enumerate(zip(cov['unknown_before_recovery'], cov['unknown_after_recovery'])):
        ax[0].text(i - 0.2, a, str(int(a)), ha='center', va='bottom', fontsize=8)
        ax[0].text(i + 0.2, b, str(int(b)), ha='center', va='bottom', fontsize=8)
    ax[0].set_xticks(x)
    ax[0].set_xticklabels(cov['field'])
    ax[0].set_ylabel('accepted V2 events with unknown value')
    ax[0].set_title('Description coverage: ' + str(int(cov['events'].iloc[0])) + ' accepted events')
    ax[0].legend(fontsize=8)
    for f in DESC_FIELDS:
        ax[1].plot(by_year['ce_year'], 100.0 * by_year['unknown_' + f] / by_year['events'], marker='o', label=f)
    ax[1].set_xticks(by_year['ce_year'])
    ax[1].set_ylabel('% unknown after recovery (blank in raw)')
    ax[1].set_title('Remaining unknown share by source year')
    ax[1].legend(fontsize=8)
    fig.suptitle('Historical description recovery (descriptive data coverage, not a risk estimate)', fontsize=10)
    fig.tight_layout()
    p = fig_dir / 'recovery_coverage.png'
    fig.savefig(p, dpi=120)
    plt.close(fig)
    out['figures/recovery_coverage.png'] = p
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
    ax[0].bar(np.arange(len(spatial_monthly)), spatial_monthly['n_selected'], color='#5a8f5a')
    ticks = [i for i, m in enumerate(spatial_monthly['year_month']) if m.endswith('-01')]
    ax[0].set_xticks(ticks)
    ax[0].set_xticklabels([spatial_monthly['year_month'].iloc[i][:4] for i in ticks])
    ax[0].set_ylabel('selected records per month')
    ax[0].set_title('Selected (50 m linked) records by month, all ' + str(len(seg)) + ' universe segments')
    v = seg['n_selected_total'].to_numpy()
    bins = np.concatenate([[-0.5, 0.5], np.unique(np.ceil(np.logspace(0, np.log10(max(v.max(), 2)), 25))) + 0.5])
    ax[1].hist(v, bins=bins, color='#a5703b')
    ax[1].set_xscale('symlog')
    ax[1].set_yscale('log')
    ax[1].set_xlabel('selected records per segment 2021-2025 (0 = empty history, not safety)')
    ax[1].set_ylabel('segments')
    ax[1].set_title(str(int((v == 0).sum())) + ' of ' + str(len(v)) + ' segments have empty selected history')
    fig.suptitle('Selected spatial history distribution (historical counts only; not a model risk forecast)', fontsize=10)
    fig.tight_layout()
    p = fig_dir / 'selected_history_distribution.png'
    fig.savefig(p, dpi=120)
    plt.close(fig)
    out['figures/selected_history_distribution.png'] = p
    return out


def build_manifest(status, cfg, cfg_sha, pins_sha, nb_rel, nb_sha, funcs_sha, cells_sha, inputs, raw_prov, report_files, data_files, env, durations,
                   checks, protected, results, started):
    return {'schema': MANIFEST_SCHEMA, 'producer': CATALOG_PRODUCER, 'version': cfg['version'], 'status': status, 'purpose': cfg['purpose'],
            'research_only': True, 'production_ready': False, 'operational_status': 'BLOCKED', 'started_utc': started, 'finished_utc': utcnow(),
            'config_path': 'config/catalog_v3.json', 'config_sha256': cfg_sha, 'manager_pins_path': cfg['pins']['path'], 'manager_pins_sha256': pins_sha,
            'notebook': nb_rel, 'notebook_source_sha256_at_start': nb_sha, 'functions_cells_sha256': funcs_sha, 'code_cells_sha256': cells_sha,
            'inputs': inputs, 'raw_sources': raw_prov, 'outputs': {'report_files': report_files, 'data_files': data_files},
            'environment': env, 'durations': durations, 'checks': checks, 'protected_snapshot': protected, 'results': results,
            'interfaces': {'load_catalog': 'load_catalog(report_dir, data_dir, config_path, expected_manifest_sha256, expected_functions_sha256, expected_config_sha256=None, source_notebook=None)',
                           'query_province_history': 'query_province_history(cat, {province_key, start_month, end_month[, categories][, top_k]})',
                           'query_spatial_history': 'query_spatial_history(cat, {segment_id, target_date, unit[, time_block (6h only, required)][, lookback_days][, categories][, top_k]})',
                           'query_spatial_history_batch': 'query_spatial_history_batch(cat, [request, ...] <= max_batch)',
                           'parse_request_json': 'parse_request_json(text, max_bytes)', 'catalog_summary': 'catalog_summary(cat)', 'list_provinces': 'list_provinces(cat)',
                           'public_result_schema': 'PUBLIC_RESULT_SCHEMA / FORBIDDEN_OUTPUT_KEYS (FUNCTIONS 4); require_public_result(res, kind) enforced inside every query',
                           'privacy_audit': 'audit_public_result(res, kind, uid_index(retained_uids)) -> {schema_violations, forbidden_keys, retained_uid_hits}'},
            'notes': [PROVINCE_NOTE, SPATIAL_NOTE, CATEGORY_NOTE, 'Descriptions are historical context only, never prospective model features.']}


def catalog_report_th(res, cov, pre):
    L = ['# รายงาน catalog ประวัติอุบัติเหตุ (notebook 14, V3)', '',
         'สถานะ: research only, production_ready = false, ไม่ใช่โมเดลหรือการพยากรณ์ และยังไม่ใช่ API', '',
         '## การกู้คำอธิบายจากไฟล์ raw', '',
         '- เหตุการณ์ V2 ที่ยอมรับแล้ว: ' + str(res['events']) + ' รายการ (legacy ตรงกัน ' + str(res['legacy_matched']) + ', กู้เพิ่มจาก raw ' + str(res['repaired_recovered']) + ')',
         '- จับคู่แบบ exact ผ่าน row_lineage (source_file/source_row) และตรวจ BE-year:ACC_CODE ทุกแถว ไม่มี fuzzy join และไม่มีแถวใหม่จาก raw',
         '- วันที่/ช่วงเวลา/จังหวัดใช้ค่า V2 ที่ซ่อมแล้ว ไม่ parse วันที่จาก raw ซ้ำ', '',
         '| field | unknown ก่อน | unknown หลัง | preflight หลัง | ขัดแย้ง legacy |', '|---|---|---|---|---|']
    for _, r in cov.iterrows():
        L.append('| ' + r['field'] + ' | ' + str(int(r['unknown_before_recovery'])) + ' | ' + str(int(r['unknown_after_recovery'])) + ' | ' +
                 str(pre['unknown_after'].get(r['field'])) + ' | ' + str(int(r['legacy_value_conflicts'])) + ' |')
    L += ['', 'ค่าที่ยังว่างคือว่างจริงในไฟล์ raw ถูกเก็บเป็น unknown ไม่เดาหรือเติมจากเหตุการณ์อื่น ตัวเลข preflight เป็นข้อมูลอ้างอิงเท่านั้น ไม่ได้ใช้แทนการตรวจจริง', '',
          '## Catalog จังหวัด', '',
          '- ' + str(res['provinces']) + ' จังหวัด x ' + str(res['months']) + ' เดือน (' + res['first_month'] + '..' + res['last_month'] + ') รวม ' + str(res['province_grid_total']) + ' รายการ',
          '- เดือนที่เป็นศูนย์ ' + str(res['zero_province_months']) + ' ช่อง หมายถึงไม่มีบันทึกในแหล่งข้อมูลนี้ ไม่ได้แปลว่าครอบคลุมถนนทั้งหมด',
          '- ชื่อและจุดของจังหวัดเป็นจุดตัวแทนเท่านั้น ไม่ใช่ตำแหน่งเหตุการณ์', '',
          '## Catalog เชิงพื้นที่ (selected ของ notebook 19)', '',
          '- selected ' + str(res['selected']) + ' รายการ บน universe ' + str(res['universe_segments']) + ' segment (มีประวัติ ' + str(res['segments_with_history']) + ' segment)',
          '- นับซ้ำ daily และ 6h ได้ตรงกับ counts ของ notebook 19 ทุกแถว',
          '- ประวัติว่าง (N = 0) หมายถึงไม่มี selected record ไม่ใช่หลักฐานว่าปลอดภัย', '',
          '## ข้อควรระวัง', '', '- ' + CATEGORY_NOTE, '- ' + SPATIAL_NOTE, '- ' + PROVINCE_NOTE, '']
    return NL.join(L)


def serving_interface_th(cfg):
    Q = cfg['query']
    return NL.join(['# อินเทอร์เฟซสำหรับ API ภายหลัง (catalog V3, research only)', '',
                    'ฟังก์ชันอยู่ใน FUNCTIONS cells ของ `notebooks/14_backend_catalog.ipynb` (API ต้อง exec cell ที่ pin hash แล้วเท่านั้น)', '',
                    '- `load_catalog(report_dir, data_dir, config_path, expected_manifest_sha256, expected_functions_sha256, expected_config_sha256=None, source_notebook=None)` '
                    'ตรวจ hash ของ manifest/config/FUNCTIONS/ทุกไฟล์ก่อนอ่าน path ต้องมาจาก server config เท่านั้น',
                    '- `query_province_history(cat, {province_key, start_month, end_month[, categories][, top_k]})` ย้อนหลัง ไม่ใช่ as-of สูงสุด ' + str(Q['max_months']) + ' เดือน',
                    '- `query_spatial_history(cat, {segment_id, target_date, unit[, time_block][, lookback_days][, categories][, top_k]})` '
                    'issue = target_date - 1 วัน 18:00 Asia/Bangkok, cutoff = issue - ' + str(Q['latency_days']) + ' วัน, นับเฉพาะ record ที่ target end <= cutoff',
                    '- `query_spatial_history_batch(cat, [..])` ไม่เกิน ' + str(Q['max_batch']) + ' requests, ห้ามซ้ำ',
                    '- `parse_request_json(text, ' + str(Q['max_request_bytes']) + ')` ปฏิเสธ duplicate keys / NaN / ขนาดเกิน', '',
                    'ข้อจำกัด: target_date ' + Q['min_target_date'] + '..' + Q['max_target_date'] + ', lookback_days <= ' + str(Q['max_lookback_days']) +
                    ', top_k <= ' + str(Q['max_top_k']) + ', type ต้องตรง (bool ไม่ใช่ int), field เกินหรือขาดถูกปฏิเสธ', '',
                    'ผลลัพธ์เป็น aggregate เท่านั้น ไม่มี event_uid, source_row, พิกัด หรือแถวเหตุการณ์ หมวดหมู่เป็นข้อมูลที่บันทึกในอดีต ไม่ใช่สาเหตุหรือคำแนะนำส่วนบุคคล',
                    'ทุก query ตรวจ key ของผลลัพธ์กับ `PUBLIC_RESULT_SCHEMA` แบบ exact (`require_public_result`) segment_id สาธารณะ (เช่น `...:doh:2577:p0:...`) อนุญาต '
                    'ส่วน event UID ตรวจด้วยการเทียบกับชุด UID จริงแบบ exact (`audit_public_result`) ไม่ใช้ regex รูปแบบ', ''])


In [ ]:
# SETUP: config, dependency check, output guard BEFORE any write, own source/code-cell hashes, protected snapshot, check/stage/put helpers.
import contextlib
import gzip
import resource

STARTED = utcnow()
cfg = read_strict_json(CONFIG_PATH)
validate_config(cfg)
CFG_SHA = sha256_file(CONFIG_PATH)
require_modules(['pandas', 'numpy', 'matplotlib'], cfg['dependency_pins'])
PATHS = output_paths(OUTPUT_ROOT, cfg)
check_outputs(PATHS, BACKEND, REPO, cfg)
NB_PATH = BACKEND / cfg['notebook']
NB_SHA = sha256_file(NB_PATH)
NB_CELLS = notebook_code_cells(NB_PATH)
FUNCS_SHA, CELLS_SHA = functions_sha(NB_CELLS), code_cells_sha(NB_CELLS)
PROTECTED_BEFORE = hash_paths(BACKEND, snapshot_rels(cfg))
CHECKS, DURATIONS, REPORT_FILES, DATA_FILES = [], {}, {}, {}
for k in ('report_dir', 'data_dir'):
    PATHS[k].mkdir(parents=True, exist_ok=False)
PATHS['figure_dir'].mkdir()
print('FUNCTIONS sha', FUNCS_SHA, '| protected files hashed', len(PROTECTED_BEFORE))


def check(name, ok, evidence=None):
    CHECKS.append({'check': name, 'status': 'pass' if ok else 'fail', 'evidence': json.dumps(jsonable(evidence), ensure_ascii=False, allow_nan=False)[:2000]})
    if not ok:
        raise CatalogError('check failed: ' + name + ' ' + str(evidence)[:500])


def peak_rss_bytes():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return int(r if platform.system() == 'Darwin' else r * 1024)


@contextlib.contextmanager
def stage(name):
    t0 = time.time()
    print('[stage]', name, flush=True)
    yield
    DURATIONS[name] = {'seconds': round(time.time() - t0, 2), 'peak_rss_bytes_so_far': peak_rss_bytes()}
    print('  done', name, DURATIONS[name], flush=True)


def put_report(name, obj):
    check_report_file(name, cfg)
    p = PATHS['report_dir'] / name
    if isinstance(obj, pd.DataFrame):
        atomic_write_bytes(p, csv_bytes(obj))
    elif isinstance(obj, (bytes, bytearray)):
        atomic_write_bytes(p, bytes(obj))
    elif isinstance(obj, str):
        atomic_write_bytes(p, obj.encode('utf-8'))
    else:
        atomic_write_json(p, obj)
    REPORT_FILES[name] = sha256_file(p)


def put_data(name, obj):
    p = PATHS['data_dir'] / name
    if isinstance(obj, pd.DataFrame):
        b = csv_bytes(obj)
        atomic_write_bytes(p, gzip.compress(b, mtime=0) if name.endswith('.gz') else b)
    else:
        atomic_write_json(p, obj)
    DATA_FILES[name] = sha256_file(p)


In [ ]:
# INPUTS: manager pins -> every pinned hash -> accepted producer membership (10 inputs/data_outputs, 19 data_files_gitignored, 19 review)
# -> 1:1 retained lineage join -> every needed raw CSV against the raw manifest and the notebook-10 v1_raw_csv record.
with stage('inputs'):
    PINS, PINS_SHA = load_pins(REPO, cfg)
    INPUT_ROWS, DOCS = verify_inputs(BACKEND, cfg, PINS)
    check('every pinned input hash-verified and corroborated by its accepted producer', len(INPUT_ROWS) == len(cfg['inputs']),
          INPUT_ROWS[['input', 'producer_membership']].to_dict('records'))
    I = cfg['inputs']
    ALIASES = DOCS['data_pipeline']['schema_aliases']
    EVENTS = read_str_csv(BACKEND / I['events_v2']['path'])
    LINEAGE = read_str_csv(BACKEND / I['row_lineage_v2']['path'])
    LEGACY = read_str_csv(BACKEND / I['legacy_descriptive']['path'])
    PROVINCES = load_provinces(read_str_csv(BACKEND / I['province_points']['path']), cfg['catalog']['expected_provinces'])
    JOINED = join_retained_lineage(EVENTS, LINEAGE, cfg)
    check('retained lineage joins accepted events 1:1 (unique UIDs, equal sets, unique integer source coordinates, same V2 date/time/province)',
          len(JOINED) == len(EVENTS), {'events': len(EVENTS), 'retained_lineage': len(JOINED)})
    RAW = verify_raw_sources(BACKEND, cfg, DOCS['raw_manifest'], DOCS['preparation_manifest'], set(JOINED['source_file']))
    check('every needed raw CSV matches the raw manifest and notebook-10 v1_raw_csv (sha, bytes, encoding, years)', len(RAW) == JOINED['source_file'].nunique(),
          {f: {'sha256': v['sha256'][:12], 'ce_year': v['entry']['ce_year'], 'rows': v['entry']['rows'], 'encoding': v['entry']['encoding']} for f, v in RAW.items()})
print(INPUT_ROWS[['input', 'sha256', 'producer_membership']].to_string(index=False))


In [ ]:
# RAW DESCRIPTION RECOVERY: exact source record per accepted event; legacy V1 descriptors must agree on every shared UID. The preflight numbers
# are printed next to the actual counts as a reference only (not a substitute check).
with stage('raw recovery'):
    def raw_loader(f):
        info = RAW[f]
        verify_pinned(info['path'], info['sha256'], 'raw ' + f)  # re-verified immediately before reading
        return read_raw_csv(info['path'], info['entry'])

    REC0, RAW_PROV = recover_descriptions(JOINED, RAW, ALIASES, raw_loader, DESC_FIELDS, int(cfg['raw']['first_data_row']))
    REC, COV, REC_SUMMARY = compare_legacy(REC0, LEGACY)
    BY_YEAR = recovery_by_year(REC)
    check('recovered rows == accepted events (no raw-only records, no row multiplication)', len(REC) == len(EVENTS) and set(REC['event_uid']) == set(EVENTS['event_uid']),
          REC_SUMMARY)
    check('zero legacy descriptor conflicts on all shared UIDs', int(COV['legacy_value_conflicts'].sum()) == 0, COV[['field', 'legacy_value_conflicts']].to_dict('records'))
    a = REC[['event_uid', 'event_date', 'time_block', 'province_key']].sort_values('event_uid').reset_index(drop=True)
    b = EVENTS[['event_uid', 'event_date', 'time_block', 'province_key']].sort_values('event_uid').reset_index(drop=True)
    check('accepted repaired date/time block/province carried unchanged', a.equals(b), None)
    PRE = cfg['recovery']['preflight_reference']
    COV['preflight_unknown_before'] = COV['field'].map(PRE['unknown_before']).astype(int)
    COV['preflight_unknown_after'] = COV['field'].map(PRE['unknown_after']).astype(int)
    COV['actual_matches_preflight'] = (COV['unknown_after_recovery'] == COV['preflight_unknown_after']) & (COV['unknown_before_recovery'] == COV['preflight_unknown_before'])
print(REC_SUMMARY)
print(COV.to_string(index=False))
print(RAW_PROV.to_string(index=False))


In [ ]:
# PROVINCE CATALOG: 77 provinces x 60 months (explicit zeros) with category marginals and known/unknown denominators.
with stage('province catalog'):
    MONTHS = month_range(cfg['catalog']['first_month'], cfg['catalog']['last_month'])
    check_event_domain(REC, PROVINCES, MONTHS)
    PM, PC = province_tables(REC, PROVINCES, MONTHS)
    PREC = reconcile_province(PM, PC, REC, PROVINCES, MONTHS)
    check('province catalog reconciles (grid, totals, years, denominators, marginals)', bool(PREC['ok'].all()), PREC[~PREC['ok']].to_dict('records'))
print(PREC.to_string(index=False))


In [ ]:
# SPATIAL CATALOG: notebook-19 selected records joined 1:1 to the recovered descriptions; daily and 6 h counts recomputed and compared with the
# accepted 19 sparse counts; every universe segment kept (empty history explicit).
with stage('spatial catalog'):
    SEL = read_str_csv(BACKEND / I['selected_events']['path'])
    UNI = read_str_csv(BACKEND / I['universe_segments']['path'])['segment_id'].tolist()
    CD = read_str_csv(BACKEND / I['counts_daily']['path'])
    C6 = read_str_csv(BACKEND / I['counts_6h']['path'])
    ROWS = spatial_rows(SEL, REC, UNI)
    SMON, STOT, SSEG, SCOV = spatial_tables(ROWS, UNI, MONTHS, cfg['catalog']['coverage_years'])
    SREC = reconcile_spatial(ROWS, SEL, UNI, CD, C6, DOCS['spatial_review'], SMON, SSEG)
    check('spatial catalog reconciles with notebook-19 selected/universe/daily/6h counts', bool(SREC['ok'].all()), SREC[~SREC['ok']].to_dict('records'))
    check('stored spatial row context carries no UID/source row/coordinates', set(ROWS.columns) == set(SPATIAL_ROW_COLS), list(ROWS.columns))
print(SREC.to_string(index=False))
print(SCOV.to_string(index=False))


In [ ]:
# EXPORT + CONTROLS: aggregate report tables (git-tracked dir) and ignored query data (no UID/source row/coordinates except the provenance
# file recovered_descriptions.csv.gz kept for the independent oracle, never served). Controls run on a view rebuilt from the WRITTEN files.
with stage('write catalog tables'):
    put_report('input_verification.csv', INPUT_ROWS)
    put_report('raw_source_provenance.csv', RAW_PROV)
    put_report('recovery_coverage.csv', COV)
    put_report('recovery_by_year.csv', BY_YEAR)
    put_report('province_monthly.csv', PM)
    put_report('province_category_monthly.csv', PC)
    put_report('province_reconciliation.csv', PREC)
    put_report('spatial_monthly.csv', SMON)
    put_report('spatial_category_totals.csv', STOT)
    put_report('spatial_coverage_by_year.csv', SCOV)
    put_report('spatial_reconciliation.csv', SREC)
    put_data('recovered_descriptions.csv.gz', REC[['event_uid', 'source_file', 'source_row', 'ce_year', 'event_date', 'time_block', 'province_key'] +
                                                  list(DESC_FIELDS) + ['legacy_status']])
    put_data('selected_history_rows.csv.gz', ROWS)
    put_data('universe_segment_ids.csv', pd.DataFrame({'segment_id': sorted(UNI)}))
    put_data('segment_history_summary.csv', SSEG)
    put_data('provinces.json', {'point_meaning': 'representative province label point (not an event or road location)',
                                'provinces': PROVINCES.to_dict('records')})

with stage('query controls on the written files'):
    for name, sha in list(REPORT_FILES.items()):
        verify_pinned(PATHS['report_dir'] / name, sha, 'written ' + name)
    for name, sha in list(DATA_FILES.items()):
        verify_pinned(PATHS['data_dir'] / name, sha, 'written ' + name)
    W_ROWS = read_str_csv(PATHS['data_dir'] / 'selected_history_rows.csv.gz')
    VIEW = build_catalog_view(cfg, read_strict_json(PATHS['data_dir'] / 'provinces.json')['provinces'], read_str_csv(PATHS['report_dir'] / 'province_monthly.csv'),
                              read_str_csv(PATHS['report_dir'] / 'province_category_monthly.csv'),
                              read_str_csv(PATHS['data_dir'] / 'universe_segment_ids.csv')['segment_id'].tolist(), W_ROWS, 'pre-manifest')
    check('written row context round-trips exactly', W_ROWS.astype(str).equals(ROWS.astype(str)), len(W_ROWS))
    CTRL = run_query_controls(VIEW, W_ROWS, cfg, REC['event_uid'])
    put_report('negative_controls.csv', CTRL)
    check('all brute-force, boundary, future-leak and strict-validation controls pass', bool(CTRL['ok'].all()), CTRL[~CTRL['ok']].to_dict('records'))
    busiest = SSEG.sort_values(['n_selected_total', 'segment_id'], ascending=[False, True]).iloc[0]['segment_id']
    empty_seg = SSEG[SSEG['n_selected_total'] == 0]['segment_id'].iloc[0] if (SSEG['n_selected_total'] == 0).any() else busiest
    EX_REQ = {'province_full_range': ('province', {'province_key': PROVINCES['province_key'].iloc[0], 'start_month': MONTHS[0], 'end_month': MONTHS[-1]}),
              'spatial_daily_busiest': ('spatial', {'segment_id': busiest, 'target_date': '2025-06-01', 'unit': 'daily'}),
              'spatial_6h_busiest': ('spatial', {'segment_id': busiest, 'target_date': '2025-06-01', 'unit': '6h', 'time_block': '18-24', 'lookback_days': 365}),
              'spatial_empty_history': ('spatial', {'segment_id': empty_seg, 'target_date': '2026-03-01', 'unit': 'daily'}),
              'spatial_before_catalog_data': ('spatial', {'segment_id': busiest, 'target_date': '2021-01-15', 'unit': 'daily'})}
    EXAMPLES = {k: {'request': r, 'response': (query_province_history(VIEW, r) if kind == 'province' else query_spatial_history(VIEW, r))}
                for k, (kind, r) in EX_REQ.items()}
    EX_AUDIT = {k: audit_public_result(EXAMPLES[k]['response'], kind, uid_index(REC['event_uid'])) for k, (kind, _) in EX_REQ.items()}
    check('query examples: explicit public schema, no forbidden key at any depth, no retained event UID',
          all(not any(a.values()) for a in EX_AUDIT.values()), EX_AUDIT)
    put_report('query_examples.json', EXAMPLES)
print(CTRL.to_string(index=False))


In [ ]:
# REPORTS + MANIFEST: figures, readiness, Thai reports, protected after-snapshot, then the manifest (status complete only after every check
# passed) and a final load_catalog with the real hashes. If that final load fails the manifest is rewritten as failed and the cell raises.
with stage('figures and reports'):
    FIGS = make_figures(PATHS['figure_dir'], COV, BY_YEAR, SMON, SSEG)
    for rel, p in FIGS.items():
        check('figure written: ' + rel, p.is_file() and p.stat().st_size > 10000, p.stat().st_size if p.is_file() else None)
        REPORT_FILES[rel] = sha256_file(p)
    RES = {'events': len(REC), 'legacy_matched': REC_SUMMARY['legacy_matched'], 'repaired_recovered': REC_SUMMARY['repaired_recovered'],
           'legacy_uids_not_accepted_events': REC_SUMMARY['legacy_uids_not_accepted_events'],
           'unknown_before': dict(zip(COV['field'], COV['unknown_before_recovery'].astype(int))), 'unknown_after': dict(zip(COV['field'], COV['unknown_after_recovery'].astype(int))),
           'actual_matches_preflight_reference': bool(COV['actual_matches_preflight'].all()),
           'provinces': len(PROVINCES), 'months': len(MONTHS), 'first_month': MONTHS[0], 'last_month': MONTHS[-1], 'province_grid_total': int(PM['n_records'].sum()),
           'zero_province_months': int((PM['n_records'] == 0).sum()), 'selected': len(ROWS), 'universe_segments': len(UNI),
           'segments_with_history': int((SSEG['n_selected_total'] > 0).sum()), 'negative_controls': len(CTRL)}
    READY = pd.DataFrame([
        {'item': 'exact raw description recovery', 'status': 'verified_by_notebook', 'note': 'awaits Codex independent raw oracle and formal review'},
        {'item': 'province retrospective catalog', 'status': 'verified_by_notebook', 'note': '77 x 60 grid; zero = no recorded records in this source'},
        {'item': 'spatial selected as-of history', 'status': 'verified_by_notebook', 'note': 'selected-conditional; cutoff issue-30d; empty history != safety'},
        {'item': 'reusable load/query functions', 'status': 'verified_by_notebook', 'note': 'FUNCTIONS 4; runtime pin of manifest/FUNCTIONS sha by manager after acceptance'},
        {'item': 'HTTP API / frontend', 'status': 'not_started', 'note': 'out of scope for notebook 14'},
        {'item': 'production readiness', 'status': 'false', 'note': 'research only; operational_status BLOCKED'}])
    put_report('readiness.csv', READY)
    put_report('catalog_report_th.md', catalog_report_th(RES, COV, PRE))
    put_report('serving_interface_th.md', serving_interface_th(cfg))

with stage('protected after-snapshot'):
    PROTECTED_AFTER = hash_paths(BACKEND, snapshot_rels(cfg))
    CHANGED = snapshot_diff(PROTECTED_BEFORE, PROTECTED_AFTER)
    put_data('protected_snapshot.json', {'paths': snapshot_rels(cfg), 'before': PROTECTED_BEFORE, 'after': PROTECTED_AFTER, 'changed': CHANGED})
    check('protected accepted inputs/outputs and read-only notebook-20 sources unchanged', not CHANGED, CHANGED[:20])
    check('source notebook unchanged during the run', sha256_file(NB_PATH) == NB_SHA, NB_SHA[:12])
    check('config unchanged during the run', sha256_file(CONFIG_PATH) == CFG_SHA, CFG_SHA[:12])
    DRIFT = [d + '/' + n for d, files in (('report_dir', REPORT_FILES), ('data_dir', DATA_FILES)) for n, s in sorted(files.items())
             if os.path.islink(PATHS[d] / n) or not (PATHS[d] / n).is_file() or sha256_file(PATHS[d] / n) != s]
    check('every written output unchanged since write', not DRIFT, {'report_files': len(REPORT_FILES), 'data_files': len(DATA_FILES), 'changed': DRIFT[:20]})

PROTECTED = {'paths': snapshot_rels(cfg), 'files_hashed': len(PROTECTED_BEFORE), 'before_sha256': canonical_sha(PROTECTED_BEFORE),
             'after_sha256': canonical_sha(PROTECTED_AFTER), 'changed': CHANGED, 'runtime_outputs_not_snapshotted': runtime_output_rels(cfg),
             'runtime_output_prefixes_not_snapshotted': cfg['runtime_output_prefixes'],
             'write_guarded_not_snapshotted': cfg['write_guard_paths'] + cfg['repo_write_guard_paths'], 'note': cfg['snapshot_note']}
RAW_SUMMARY = [{k: v for k, v in r.items()} for r in RAW_PROV.to_dict('records')]


def _manifest(status, extra=None):
    m = build_manifest(status, cfg, CFG_SHA, PINS_SHA, cfg['notebook'], NB_SHA, FUNCS_SHA, CELLS_SHA, INPUT_ROWS.to_dict('records'), RAW_SUMMARY,
                       dict(REPORT_FILES), dict(DATA_FILES), environment(), DURATIONS, CHECKS, PROTECTED, RES, STARTED)
    m.update(extra or {})
    return m


MAN_PATH = PATHS['report_dir'] / 'catalog_manifest.json'
atomic_write_json(MAN_PATH, _manifest('complete'))
MAN_SHA = sha256_file(MAN_PATH)
try:
    CAT = load_catalog(PATHS['report_dir'], PATHS['data_dir'], CONFIG_PATH, MAN_SHA, FUNCS_SHA, CFG_SHA, NB_PATH)
    for k, (kind, r) in EX_REQ.items():
        got = query_province_history(CAT, r) if kind == 'province' else query_spatial_history(CAT, r)
        if {a: b for a, b in got.items() if a != 'catalog_id'} != {a: b for a, b in EXAMPLES[k]['response'].items() if a != 'catalog_id'}:
            raise CatalogError('loaded catalog answers differ from the pre-manifest view: ' + k)
except Exception as e:
    atomic_write_json(MAN_PATH, _manifest('failed', {'final_load_error': type(e).__name__ + ': ' + str(e)[:1000]}))
    raise
print('catalog complete | manifest sha256', MAN_SHA, '| FUNCTIONS sha256', FUNCS_SHA)
print(json.dumps(catalog_summary(CAT), ensure_ascii=False, indent=1)[:3000])


## Outputs

`reports/v3/historical_catalog/` (aggregates only): `catalog_manifest.json` (status `complete` only after every check), `input_verification.csv`,
`raw_source_provenance.csv`, `recovery_coverage.csv`, `recovery_by_year.csv`, `province_monthly.csv`, `province_category_monthly.csv`,
`province_reconciliation.csv`, `spatial_monthly.csv`, `spatial_category_totals.csv`, `spatial_coverage_by_year.csv`, `spatial_reconciliation.csv`,
`negative_controls.csv`, `query_examples.json`, `readiness.csv`, `catalog_report_th.md`, `serving_interface_th.md`,
`figures/recovery_coverage.png`, `figures/selected_history_distribution.png`.

`data/v3/historical_catalog/` (gitignored): `selected_history_rows.csv.gz` (segment/date/block/descriptions; no UID, source row or coordinates),
`universe_segment_ids.csv`, `segment_history_summary.csv`, `provinces.json` (representative points), `protected_snapshot.json` and
`recovered_descriptions.csv.gz` (UID + source coordinates: provenance for the independent oracle only, never served).

Later API: exec the pinned FUNCTIONS cells, then `load_catalog(report_dir, data_dir, config_path, manifest_sha, functions_sha, ...)` and
`query_province_history` / `query_spatial_history` / `query_spatial_history_batch`. A missing manifest or `status != complete` means the run failed.
